In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial.distance import cosine, pdist
from scipy import stats

In [81]:
CT_COL     = "TCRClonotype"
SAMPLE_COL = "Sample_Origin"
LY49C_COL  = "Ly-49C-Ly-49I-Klra3-Klra9-AMM2139-pAbO"
SYFPEITHI_COL = "RiO-H-2:H-2Kd-SYFPEITHI-ADEX5099-pAbO"

min_size = 5
plots_per_page = 16
thresholds = np.linspace(1.0, 0.00, 201)
default_q = 0.925

DEBUG_PROBS = False

In [82]:
markers_b10br = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-EEEPVKKI-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
]

peptides_b10br = [
    "ATLVFHNL","EEEPVKKI","HIYEFPQL","INFDFPKL","RAYLFNSV","RTYTYEKL",
    "SNYLFTKL","SSYTFPKM","SVYVYKVL","VAFDFTKV","VGPRYTNL","VIVRFLTV","VSFTYRYL"
]

markers_balbc = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
    "RiO-H-2:H-2Kd-SYFPEITHI-ADEX5099-pAbO"
]

peptides_balbc = [
    "ATLVFHNL","HIYEFPQL","INFDFPKL","RAYLFNSV","RTYTYEKL","SNYLFTKL",
    "SSYTFPKM","SVYVYKVL","VAFDFTKV","VGPRYTNL","VIVRFLTV","VSFTYRYL","SYFPEITHI"
]

In [83]:
def filter_zero_dextramer_cells(
    df: pd.DataFrame,
    markers: list[str],
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Remove cells (rows) where the sum of all dextramer marker counts is zero.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with cells as rows
    markers : list[str]
        List of dextramer marker column names
    verbose : bool
        If True, print filtering statistics
        
    Returns
    -------
    pd.DataFrame
        Filtered dataframe with zero-dextramer cells removed
    """
    dex_sum = df[markers].sum(axis=1)
    mask = dex_sum > 0
    n_before = len(df)
    n_after = int(mask.sum())
    n_removed = n_before - n_after
    
    if verbose:
        pct_removed = 100.0 * n_removed / n_before if n_before > 0 else 0.0
        print(f"[filter_zero_dextramer] Removed {n_removed:,} / {n_before:,} cells "
              f"({pct_removed:.2f}%) with zero total dextramer counts")
    
    return df[mask].copy()

In [84]:
def row_normalise(
    X: np.ndarray,
    eps: float = 0.0,
    debug: bool = False,
    tag: str = "P",
) -> np.ndarray:
    """
    Row-normalise raw peptide intensities/counts -> probabilities per cell.

    If a row sums to 0 (all peptides 0), we return an all-zero probability row.
    Those contribute 0 entropy under our convention (see renyi_entropy).
    """
    X = np.asarray(X, dtype=float)
    rs = X.sum(axis=1, keepdims=True)

    zero = (rs.squeeze() <= 0)
    if debug:
        frac = 100.0 * float(np.mean(zero)) if X.shape[0] else 0.0
        print(f"[DEBUG] {tag}: {frac:.2f}% cells have zero RiO mass (all peptides 0).")

    rs_safe = rs.copy()
    rs_safe[rs_safe <= 0] = 1.0
    P = X / rs_safe

    if eps > 0:
        P = np.maximum(P, eps)
        P = P / P.sum(axis=1, keepdims=True)

    return P

In [85]:
def renyi_entropy(P: np.ndarray, alpha: float, axis: int = -1) -> np.ndarray:
    """
    Renyi entropy in bits for probability vectors P.

    IMPORTANT: Handles all-zero rows safely (returns 0 for those rows).
      alpha=1 -> Shannon
      alpha=0 -> Hartley (log2 support size)
    """
    P = np.asarray(P, dtype=float)
    s = P.sum(axis=axis)
    zero_mask = (s == 0)

    def apply_zero_mask(H):
        if np.isscalar(H):
            return 0.0 if zero_mask else H
        H = np.asarray(H, dtype=float)
        H[zero_mask] = 0.0
        return H

    if alpha == 1.0:
        with np.errstate(divide="ignore", invalid="ignore"):
            logP = np.where(P > 0, np.log2(P), 0.0)
        H = -(P * logP).sum(axis=axis)
        return apply_zero_mask(H)

    if alpha == 0.0:
        k = np.sum(P > 0, axis=axis)
        k = np.maximum(k, 1)
        H = np.log2(k)
        return apply_zero_mask(H)

    S = np.sum(np.power(np.maximum(P, 0.0), alpha), axis=axis)
    S = np.maximum(S, 1e-300)
    H = (1.0 / (1.0 - alpha)) * np.log2(S)
    return apply_zero_mask(H)

In [86]:
def mean_pairwise_cosine_similarity(P: np.ndarray) -> float:
    """
    P: row-normalised per cell (n_cells x n_peptides).
    Returns mean pairwise cosine similarity, or NaN if <2 cells.
    """
    nonzero_mask = P.sum(axis=1) > 0
    P = P[nonzero_mask]
    if P.shape[0] < 2:
        return np.nan
    return float(np.mean(1.0 - pdist(P, metric="cosine")))

In [87]:
def compute_clonotype_summaries(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    entropy_alpha: float = 1.0,
    debug_probs: bool = False,
) -> dict:
    """
    Returns dict ct -> summary:
      - mean_pattern: mean of per-cell normalised vectors
      - mean_entropy: mean Renyi entropy (alpha) across cells
      - mean_coherence: mean pairwise cosine similarity across cells
      - n_cells: number of cells
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    out = {}

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")

        ent = float(np.mean(renyi_entropy(P, alpha=float(entropy_alpha), axis=1)))
        coh = mean_pairwise_cosine_similarity(P)

        out[ct] = {
            "mean_pattern": P.mean(axis=0),
            "mean_entropy": ent,
            "mean_coherence": coh,
            "n_cells": int(P.shape[0]),
        }
    return out

In [88]:
def apply_ly49c_filter_one_sample(df_s: pd.DataFrame, ly49c_col: str, q: float) -> tuple[pd.DataFrame, float]:
    """
    Within ONE sample: keep Ly49C <= quantile(q).
    Returns (filtered_df, Ts).
    """
    if q >= 1.0:
        return df_s.copy(), float(df_s[ly49c_col].max())
    Ts = float(df_s[ly49c_col].quantile(q))
    return df_s[df_s[ly49c_col] <= Ts].copy(), Ts

In [89]:
def apply_reverse_filter_one_sample(df_s: pd.DataFrame, col: str, q: float) -> tuple[pd.DataFrame, float]:
    """
    Within ONE sample: keep cells where col >= (1-q) percentile.
    At q=1.0, keeps 100% of cells (no filter).
    At q=0.99, keeps top 99% (removes bottom 1%).
    Returns (filtered_df, Ts) where Ts is the lower bound threshold.
    """
    if q >= 1.0:
        return df_s.copy(), float(df_s[col].min())
    lower_bound = float(df_s[col].quantile(1.0 - q))
    return df_s[df_s[col] >= lower_bound].copy(), lower_bound

In [90]:
def pattern_distance(p1: np.ndarray, p2: np.ndarray, metric: str) -> float:
    if metric == "cosine":
        if np.allclose(p1, 0) or np.allclose(p2, 0):
            return 1.0
        return float(cosine(p1, p2))
    if metric == "l1":
        return float(np.sum(np.abs(p1 - p2)))
    raise ValueError(f"Unknown metric: {metric}")

In [91]:
def ly49c_sweep_metrics_one_sample(
    df_s: pd.DataFrame,
    ct_col: str,
    ly49c_col: str,
    markers: list[str],
    thresholds: np.ndarray,
    min_size: int = 5,
    entropy_alpha: float = 1.0,
    debug_probs: bool = False,
) -> pd.DataFrame:
    """
    Computes sweep metrics for ONE sample (entropy order fixed).

    Outputs per q:
      - retention
      - median cosine dist between mean clonotype patterns (baseline vs filtered)
      - median L1 dist
      - mean entropy change (filtered - baseline) across common clonotypes
      - mean coherence change (filtered - baseline)
    """
    baseline = compute_clonotype_summaries(
        df_s, ct_col, markers, min_size=min_size, entropy_alpha=entropy_alpha, debug_probs=debug_probs
    )
    n0 = len(df_s)

    rows = []
    for q in thresholds:
        df_f, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, float(q))
        filtered = compute_clonotype_summaries(
            df_f, ct_col, markers, min_size=min_size, entropy_alpha=entropy_alpha, debug_probs=debug_probs
        )

        common = set(baseline) & set(filtered)
        if common:
            cos_d = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "cosine") for c in common]
            l1_d  = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "l1") for c in common]
            ent_d = [filtered[c]["mean_entropy"] - baseline[c]["mean_entropy"] for c in common]
            coh_d = [filtered[c]["mean_coherence"] - baseline[c]["mean_coherence"] for c in common]

            row = {
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "n_cells": int(len(df_f)),
                "pct_cells_retained": 100.0 * len(df_f) / n0 if n0 else np.nan,
                "n_clonotypes_baseline_ge5": int(len(baseline)),
                "n_clonotypes_filtered_ge5": int(len(filtered)),
                "n_common_clonotypes_ge5": int(len(common)),
                "median_cosine_dist": float(np.median(cos_d)),
                "median_l1_dist": float(np.median(l1_d)),
                "mean_entropy_change": float(np.mean(ent_d)),
                "mean_coherence_change": float(np.nanmean(coh_d)),
            }
        else:
            row = {
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "n_cells": int(len(df_f)),
                "pct_cells_retained": 100.0 * len(df_f) / n0 if n0 else np.nan,
                "n_clonotypes_baseline_ge5": int(len(baseline)),
                "n_clonotypes_filtered_ge5": int(len(filtered)),
                "n_common_clonotypes_ge5": 0,
                "median_cosine_dist": np.nan,
                "median_l1_dist": np.nan,
                "mean_entropy_change": np.nan,
                "mean_coherence_change": np.nan,
            }

        rows.append(row)

    return pd.DataFrame(rows)

In [92]:
def ly49c_sweep_entropy_orders_one_sample(
    df_s: pd.DataFrame,
    sample_name: str,
    ct_col: str,
    ly49c_col: str,
    markers: list[str],
    thresholds: np.ndarray,
    orders: list[float],
    min_size: int = 5,
    debug_probs: bool = False,
) -> pd.DataFrame:
    """
    Long-form table for entropy change vs q and alpha, computed over common clonotypes.

    Returns columns:
      sample, percentile, Ts_ly49c, alpha, mean_entropy_change, n_common_clonotypes_ge5
    """
    base_summ = {}
    vc = df_s[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df_s[df_s[ct_col].isin(keep)]

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")
        base_summ[ct] = {a: float(np.mean(renyi_entropy(P, alpha=float(a), axis=1))) for a in orders}

    rows = []
    for q in thresholds:
        df_f, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, float(q))

        vc_f = df_f[ct_col].value_counts()
        keep_f = vc_f[vc_f >= min_size].index
        df_f2 = df_f[df_f[ct_col].isin(keep_f)]

        filt_summ = {}
        for ct, g in df_f2.groupby(ct_col):
            X = g[markers].to_numpy(dtype=float, copy=False)
            P = row_normalise(X, debug=debug_probs, tag=f"P_all[{ct}]")
            filt_summ[ct] = {a: float(np.mean(renyi_entropy(P, alpha=float(a), axis=1))) for a in orders}

        common = sorted(set(base_summ) & set(filt_summ))
        for a in orders:
            if common:
                dH = [filt_summ[ct][a] - base_summ[ct][a] for ct in common]
                mean_dH = float(np.mean(dH))
                n_common = int(len(common))
            else:
                mean_dH = np.nan
                n_common = 0

            rows.append({
                "sample": sample_name,
                "percentile": float(q),
                "Ts_ly49c": float(Ts),
                "alpha": float(a),
                "mean_entropy_change": mean_dH,
                "n_common_clonotypes_ge5": n_common,
                "n_cells_after": int(len(df_f)),
            })

    return pd.DataFrame(rows)

In [93]:
def clonotype_raw_and_prop_from_raw(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
) -> tuple[dict, dict, dict]:
    """
    For clonotypes with >=min_size cells:
      raw_mean[ct] = mean of raw peptide counts per cell
      prop[ct]     = raw_mean / raw_mean.sum() (i.e., normalized pattern)
      n_cells[ct]  = number of cells in clonotype (in THIS df)
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]

    raw_mean, prop = {}, {}
    n_cells = {k: int(v) for k, v in vc[keep].to_dict().items()}

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        m = X.mean(axis=0)
        raw_mean[ct] = m
        tot = m.sum()
        prop[ct] = m / tot if tot > 0 else np.zeros_like(m)

    return raw_mean, prop, n_cells

In [94]:
def plot_raw_and_prop_before_after_4x8_pdf(
    raw_b: dict, raw_a: dict,
    prop_b: dict, prop_a: dict,
    n_b: dict, n_a: dict,
    peptides: list[str],
    out_pdf: Path,
    title: str,
):
    xs = np.arange(len(peptides))
    w = 0.42
    per_page = 16

    clonotypes = sorted(n_b.keys(), key=lambda c: n_b[c], reverse=True)

    with PdfPages(out_pdf) as pdf:
        for start in range(0, len(clonotypes), per_page):
            chunk = clonotypes[start:start + per_page]

            fig, axes = plt.subplots(4, 8, figsize=(24, 12))
            axes = axes.flatten()

            for i, ct in enumerate(chunk):
                ax_raw = axes[2*i]
                ax_prp = axes[2*i + 1]
                has_after = ct in raw_a

                # Mean raw counts per cell
                b = raw_b[ct]
                if has_after:
                    a = raw_a[ct]
                    ax_raw.bar(xs - w/2, b, w, alpha=0.85, label="Before")
                    ax_raw.bar(xs + w/2, a, w, alpha=0.85, label="After")
                    ax_raw.set_title(f"{ct}\nMEAN n:{n_b[ct]}→{n_a.get(ct,0)}", fontsize=7)
                else:
                    ax_raw.bar(xs, b, w*1.8, alpha=0.6, label="Before (lost)")
                    ax_raw.set_title(f"{ct}\nMEAN n:{n_b[ct]}→0 LOST", fontsize=7, color="darkred")

                ax_raw.set_xticks(xs)
                ax_raw.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax_raw.tick_params(axis="y", labelsize=6)
                ax_raw.spines["top"].set_visible(False)
                ax_raw.spines["right"].set_visible(False)
                if i == 0:
                    ax_raw.legend(fontsize=7, loc="upper right")

                # Proportions
                b = prop_b[ct]
                if has_after:
                    a = prop_a[ct]
                    ax_prp.bar(xs - w/2, b, w, alpha=0.85, label="Before")
                    ax_prp.bar(xs + w/2, a, w, alpha=0.85, label="After")
                    ax_prp.set_title(f"{ct}\nPROP n:{n_b[ct]}→{n_a.get(ct,0)}", fontsize=7)
                else:
                    ax_prp.bar(xs, b, w*1.8, alpha=0.6, label="Before (lost)")
                    ax_prp.set_title(f"{ct}\nPROP n:{n_b[ct]}→0 LOST", fontsize=7, color="darkred")

                ax_prp.set_xticks(xs)
                ax_prp.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax_prp.tick_params(axis="y", labelsize=6)
                ax_prp.spines["top"].set_visible(False)
                ax_prp.spines["right"].set_visible(False)

            for j in range(2 * len(chunk), 32):
                axes[j].axis("off")

            page = start // per_page + 1
            n_pages = (len(clonotypes) + per_page - 1) // per_page
            fig.suptitle(f"{title} (page {page}/{n_pages})", fontsize=14, y=0.995)
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pdf.savefig(fig)
            plt.close(fig)

In [95]:
def renyi_profile_all_clonotypes_ge5(
    df_s: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    orders: list[float] | None = None,
    debug: bool = False,
) -> pd.DataFrame:
    """
    For ALL clonotypes with >= min_size cells in ONE sample:
      compute mean Renyi entropy per clonotype for each alpha in `orders`.

    Returns long-form: clonotype, n_cells, alpha, renyi_entropy_mean
    """
    if orders is None:
        orders = [0, 1, 2, 3, 4, 5]

    vc = df_s[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df_s[df_s[ct_col].isin(keep)]

    rows = []
    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X, debug=debug, tag=f"P_all[{ct}]")
        n_cells = int(P.shape[0])

        for a in orders:
            h = float(np.mean(renyi_entropy(P, alpha=float(a), axis=1)))
            rows.append({"clonotype": ct, "n_cells": n_cells, "alpha": float(a), "renyi_entropy_mean": h})

    return pd.DataFrame(rows)

In [96]:
def plot_renyi_spaghetti_all_clonotypes(
    df_long: pd.DataFrame,
    out_png: Path,
    title: str,
    alpha_linewidth: float = 1.7,
    alpha_opacity: float = 0.55,
):
    """
    df_long columns: clonotype, alpha, renyi_entropy_mean
    Plots one line per clonotype (spaghetti), darker & thicker.
    """
    fig, ax = plt.subplots(figsize=(8.2, 6.2))

    if df_long.empty or "clonotype" not in df_long.columns:
        ax.text(0.5, 0.5, "No clonotypes ≥ min_size", 
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_xlabel("Renyi order α")
        ax.set_ylabel("Mean Renyi entropy (bits)")
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(out_png, dpi=170, bbox_inches="tight")
        plt.close(fig)
        return

    for ct, g in df_long.groupby("clonotype"):
        g2 = g.sort_values("alpha")
        ax.plot(
            g2["alpha"], g2["renyi_entropy_mean"],
            marker="o",
            lw=alpha_linewidth,
            alpha=alpha_opacity
        )

    ax.set_xlabel("Renyi order α")
    ax.set_ylabel("Mean Renyi entropy (bits)")
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [97]:
def plot_entropy_sweep_overlay_alpha_by_sample_2x3(
    df_long: pd.DataFrame,
    out_png: Path,
    title: str,
    chosen_q: dict[str, float] | None = None,
    max_panels: int = 6,
):
    """
    Expects df_long columns: sample, percentile, alpha, mean_entropy_change
    Produces 2x3 grid of samples; within each panel overlays α curves.
    If chosen_q is a dict, draws vertical line at sample-specific threshold.
    """
    samples = list(df_long["sample"].dropna().unique())[:max_panels]

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
    axes = axes.flatten()

    for i in range(6):
        ax = axes[i]
        if i >= len(samples):
            ax.axis("off")
            continue

        s = samples[i]
        sub = df_long[df_long["sample"] == s].copy()
        for a, g in sub.groupby("alpha"):
            g2 = g.sort_values("percentile")
            ax.plot(g2["percentile"], g2["mean_entropy_change"], marker="o", lw=1.6, alpha=0.9, label=f"α={a:g}")

        if chosen_q is not None and s in chosen_q:
            ax.axvline(chosen_q[s], color="red", ls="--", alpha=0.7)

        ax.set_title(str(s))
        ax.set_xlabel("q (Ly49C percentile)")
        ax.invert_xaxis()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(frameon=False, fontsize=8, ncol=2)

    fig.suptitle(title, y=0.98)
    fig.text(0.5, 0.04, "q (Ly49C percentile, within sample)", ha="center")
    fig.text(0.04, 0.5, "Mean ΔHα (after − before) [bits]", va="center", rotation="vertical")
    fig.tight_layout(rect=[0.06, 0.06, 1, 0.95])
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [98]:
def coherence_before_after_table(
    df_before: pd.DataFrame,
    df_after: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    debug_probs: bool = False,
) -> pd.DataFrame:
    base = compute_clonotype_summaries(
        df_before, ct_col, markers, min_size=min_size, entropy_alpha=1.0, debug_probs=debug_probs
    )
    aft = compute_clonotype_summaries(
        df_after, ct_col, markers, min_size=min_size, entropy_alpha=1.0, debug_probs=debug_probs
    )

    common = sorted(set(base) & set(aft))
    
    if not common:
        return pd.DataFrame(columns=["clonotype", "coh_before", "coh_after", "n_before", "n_after"])
    
    rows = []
    for ct in common:
        rows.append({
            "clonotype": ct,
            "coh_before": float(base[ct]["mean_coherence"]),
            "coh_after": float(aft[ct]["mean_coherence"]),
            "n_before": int(base[ct]["n_cells"]),
            "n_after": int(aft[ct]["n_cells"]),
        })
    return pd.DataFrame(rows)

In [99]:
def plot_coherence_before_after_scatter(
    df_pairs: pd.DataFrame,
    out_png: Path,
    title: str,
    mean_coherence_change: float | None = None,
):
    """
    Scatter plot of coherence before vs after filtering.
    Includes mean coherence change annotation if provided.
    """
    fig, ax = plt.subplots(figsize=(6.5, 6.5))

    if df_pairs.empty or "coh_before" not in df_pairs.columns:
        ax.text(0.5, 0.5, "No common clonotypes ≥ min_size", 
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_xlabel("Mean coherence (baseline)")
        ax.set_ylabel("Mean coherence (filtered)")
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(out_png, dpi=170, bbox_inches="tight")
        plt.close(fig)
        return

    x = df_pairs["coh_before"].to_numpy()
    y = df_pairs["coh_after"].to_numpy()

    ax.scatter(x, y, s=18, alpha=0.7)

    lo = np.nanmin(np.r_[x, y])
    hi = np.nanmax(np.r_[x, y])
    ax.plot([lo, hi], [lo, hi], ls="--", color="black", alpha=0.6)

    ax.set_xlabel("Mean coherence (baseline)")
    ax.set_ylabel("Mean coherence (filtered)")
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    annot_lines = []
    if len(df_pairs) >= 3 and np.isfinite(x).all() and np.isfinite(y).all():
        r = float(np.corrcoef(x, y)[0, 1])
        annot_lines.append(f"r = {r:.2f}")
    annot_lines.append(f"N = {len(df_pairs)}")
    
    if mean_coherence_change is not None and np.isfinite(mean_coherence_change):
        sign = "+" if mean_coherence_change >= 0 else ""
        annot_lines.append(f"Mean Δcoh = {sign}{mean_coherence_change:.3f}")
    
    ax.text(0.02, 0.98, "\n".join(annot_lines), transform=ax.transAxes,
            va="top", ha="left", fontsize=10)

    fig.tight_layout()
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [100]:
def plot_combined_coherence(
    before_dfs: dict[str, pd.DataFrame],
    after_dfs: dict[str, pd.DataFrame],
    samples_to_combine: list[str],
    markers: list[str],
    output_path: Path,
    title: str,
    ct_col: str = CT_COL,
    min_size: int = 5,
    debug_probs: bool = False,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Combine pre-filtered dataframes and compute coherence before vs after.
    
    Parameters
    ----------
    before_dfs : dict[str, pd.DataFrame]
        Sample -> dataframe before Ly49C filtering
    after_dfs : dict[str, pd.DataFrame]
        Sample -> dataframe after Ly49C filtering
    samples_to_combine : list[str]
        List of sample names to combine
    markers : list[str]
        Dextramer marker columns
    output_path : Path
        Full path for output files (without extension)
    title : str
        Plot title
    
    Returns
    -------
    pd.DataFrame
        Coherence pairs table
    """
    df_before = pd.concat([before_dfs[s] for s in samples_to_combine], ignore_index=True)
    df_after = pd.concat([after_dfs[s] for s in samples_to_combine], ignore_index=True)
    
    if verbose:
        print(f"  Combined: {len(df_before):,} -> {len(df_after):,} cells")
    
    pairs = coherence_before_after_table(
        df_before=df_before,
        df_after=df_after,
        ct_col=ct_col,
        markers=markers,
        min_size=min_size,
        debug_probs=debug_probs,
    )
    
    if not pairs.empty:
        mean_coh_change = float((pairs["coh_after"] - pairs["coh_before"]).mean())
    else:
        mean_coh_change = None
    
    output_path = Path(output_path)
    pairs.to_csv(output_path.with_suffix(".csv"), index=False)
    
    plot_coherence_before_after_scatter(
        pairs,
        out_png=output_path.with_suffix(".png"),
        title=title,
        mean_coherence_change=mean_coh_change,
    )
    
    if verbose:
        print(f"  Saved: {output_path.with_suffix('.png')}")
        if mean_coh_change is not None:
            print(f"  Mean Δcoherence: {mean_coh_change:+.4f}")
        print(f"  N clonotypes ≥{min_size}: {len(pairs)}")
    
    return pairs

In [101]:
def plot_all_clonotypes_by_coherence_pdf(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    out_pdf: Path,
    title_prefix: str = "",
    min_size: int = 5,
    normalize: bool = True,
    jitter_width: float = 0.25,
    plots_per_page: int = 12,
    debug_probs: bool = False,
    rank_map: dict | None = None
):
    """
    Multi-page PDF with one panel per clonotype, ordered by decreasing coherence.
    
    Parameters
    ----------
    df : pd.DataFrame
        Data containing cells
    ct_col : str
        Column name for clonotype
    markers : list[str]
        Dextramer marker column names
    peptides : list[str]
        Short peptide names for x-axis labels
    out_pdf : Path
        Output PDF path
    title_prefix : str
        Prefix for the PDF suptitle (e.g., dataset/sample name)
    min_size : int
        Minimum cells per clonotype to include
    normalize : bool
        If True, plot proportions. If False, plot raw counts.
    plots_per_page : int
        Number of clonotype panels per page (e.g., 12 = 3x4 grid)
    """
    out_pdf = Path(out_pdf)
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    
    # Compute coherence for all clonotypes >= min_size
    summaries = compute_clonotype_summaries(
        df, ct_col, markers, min_size=min_size, entropy_alpha=1.0, debug_probs=debug_probs
    )
    
    if not summaries:
        print("No clonotypes >= min_size")
        return
    
    # Build list sorted by decreasing coherence
    coh_list = [
        (ct, s["mean_coherence"], s["n_cells"]) 
        for ct, s in summaries.items() 
        if np.isfinite(s["mean_coherence"])
    ]
    coh_list.sort(key=lambda x: x[1], reverse=True)
    
    n_clonotypes = len(coh_list)
    print(f"Plotting {n_clonotypes} clonotypes ordered by decreasing coherence")
    
    if plots_per_page == 12:
        nrows, ncols = 3, 4
    elif plots_per_page == 16:
        nrows, ncols = 4, 4
    elif plots_per_page == 9:
        nrows, ncols = 3, 3
    else:
        ncols = 4
        nrows = (plots_per_page + ncols - 1) // ncols
    
    n_pages = (n_clonotypes + plots_per_page - 1) // plots_per_page
    
    ylabel = "Proportion" if normalize else "Raw count"
    xs = np.arange(len(peptides))
    
    with PdfPages(out_pdf) as pdf:
        for page_idx in range(n_pages):
            start = page_idx * plots_per_page
            end = min(start + plots_per_page, n_clonotypes)
            chunk = coh_list[start:end]
            
            fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
            axes = axes.flatten()
            
            for i, (ct, coh, n_cells) in enumerate(chunk):
                ax = axes[i]
                
                df_ct = df[df[ct_col] == ct]
                X = df_ct[markers].to_numpy(dtype=float)
                
                if normalize:
                    row_sums = X.sum(axis=1, keepdims=True)
                    row_sums[row_sums == 0] = 1.0
                    X = X / row_sums
                
                for j in range(X.shape[0]):
                    jitter = np.random.uniform(-jitter_width, jitter_width, len(peptides))
                    ax.scatter(xs + jitter, X[j, :], alpha=0.5, s=15, edgecolors="none")
                
                mean_vals = X.mean(axis=0)
                ax.scatter(xs, mean_vals, marker="D", s=40, color="black", zorder=10)
                
                ax.set_xticks(xs)
                ax.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax.tick_params(axis="y", labelsize=7)
                ax.set_ylabel(ylabel, fontsize=7)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                
                rank = rank_map[ct] if rank_map and ct in rank_map else start + i + 1
                ax.set_title(f"#{rank} {ct}\ncoh={coh:.3f}, n={n_cells}", fontsize=8)
            
            for j in range(len(chunk), len(axes)):
                axes[j].axis("off")
            
            fig.suptitle(
                f"{title_prefix} | Clonotypes by coherence (page {page_idx + 1}/{n_pages})",
                fontsize=12, y=0.995
            )
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pdf.savefig(fig)
            plt.close(fig)
    
    print(f"Saved: {out_pdf}")

In [102]:
def plot_sweep_metrics_2x3(
    sweep_df: pd.DataFrame,
    sweep_entropy_df: pd.DataFrame,
    out_png: Path,
    title: str,
    sample_q: float | None = None,
):
    """
    2x3 panel plot showing sweep metrics:
      [0,0] Median cosine distance
      [0,1] Median L1 distance
      [0,2] Mean entropy change (α=0,1,2,3,4,5 overlaid)
      [1,0] Mean coherence change
      [1,1] Data retention (% cells retained)
      [1,2] Number of common clonotypes ≥ min_size
    """
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
    
    q_vals = sweep_df["percentile"].to_numpy()
    
    # [0,0] Median cosine distance
    ax = axes[0, 0]
    ax.plot(q_vals, sweep_df["median_cosine_dist"], marker="o", lw=1.8, color="C0")
    ax.set_ylabel("Median cosine distance")
    ax.set_title("Pattern shift (cosine)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    ax.invert_xaxis()
    
    # [0,1] Median L1 distance
    ax = axes[0, 1]
    ax.plot(q_vals, sweep_df["median_l1_dist"], marker="o", lw=1.8, color="C1")
    ax.set_ylabel("Median L1 distance")
    ax.set_title("Pattern shift (L1)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    # [0,2] Mean entropy change (α=0..5 overlaid)
    ax = axes[0, 2]
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, 6))
    for i, alpha_val in enumerate([0, 1, 2, 3, 4, 5]):
        sub = sweep_entropy_df[sweep_entropy_df["alpha"] == alpha_val].sort_values("percentile")
        if not sub.empty:
            ax.plot(sub["percentile"], sub["mean_entropy_change"], 
                   marker="o", lw=1.5, alpha=0.85, color=colors[i], label=f"α={alpha_val}")
    ax.set_ylabel("Mean ΔH (bits)")
    ax.set_title("Entropy change by α")
    ax.legend(frameon=False, fontsize=8, ncol=2, loc="best")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    # [1,0] Mean coherence change
    ax = axes[1, 0]
    ax.plot(q_vals, sweep_df["mean_coherence_change"], marker="o", lw=1.8, color="C2")
    ax.axhline(0, color="gray", ls=":", alpha=0.5)
    ax.set_ylabel("Mean Δcoherence")
    ax.set_title("Coherence change")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    # [1,1] Data retention
    ax = axes[1, 1]
    ax.plot(q_vals, sweep_df["pct_cells_retained"], marker="o", lw=1.8, color="C3")
    ax.set_ylabel("% cells retained")
    ax.set_title("Data retention")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    # [1,2] Number of common clonotypes
    ax = axes[1, 2]
    ax.plot(q_vals, sweep_df["n_common_clonotypes_ge5"], marker="o", lw=1.8, color="C4")
    ax.set_ylabel("N common clonotypes ≥5")
    ax.set_title("Clonotype retention")
    ax.set_xlabel("q (Ly49C percentile)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if sample_q is not None:
        ax.axvline(sample_q, color="red", ls="--", alpha=0.7)
    
    fig.suptitle(title, fontsize=12, y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [103]:
def run_samplewise_pipeline(
    df: pd.DataFrame,
    dataset_name: str,
    markers: list[str],
    peptides: list[str],
    output_root: Path,
    thresholds: np.ndarray,
    chosen_q: dict[str, float],
    ct_col: str = CT_COL,
    sample_col: str = SAMPLE_COL,
    ly49c_col: str = LY49C_COL,
    min_size: int = 5,
    entropy_alpha_for_sweep: float = 1.0,
    renyi_orders: list[float] | None = None,
    debug_probs: bool = False,
    filter_zero_dextramer: bool = True,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], dict[str, pd.DataFrame]]:
    """
    Parameters
    ----------
    chosen_q : dict[str, float]
        Maps sample name -> Ly49C percentile threshold.
        All samples in the data must have an entry (raises KeyError otherwise).
    
    Returns
    -------
    index_df : pd.DataFrame
        Index of all outputs
    before_dfs : dict[str, pd.DataFrame]
        Sample name -> dataframe before Ly49C filtering (after zero-dex removal)
    after_dfs : dict[str, pd.DataFrame]
        Sample name -> dataframe after Ly49C filtering
    """
    output_root = Path(output_root)
    ds_out = output_root / dataset_name
    ds_out.mkdir(parents=True, exist_ok=True)

    orders = renyi_orders or [0, 1, 2, 3, 4, 5]

    needed = {ct_col, sample_col, ly49c_col, *markers}
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns (first few): {missing[:10]}")

    if filter_zero_dextramer:
        if verbose:
            print(f"Dataset: {dataset_name}")
        df = filter_zero_dextramer_cells(df, markers, verbose=verbose)

    index_rows = []
    sweep_entropy_all_samples = []
    before_dfs = {}
    after_dfs = {} 

    for sample, df_s in df.groupby(sample_col):
        s_out = ds_out / str(sample)
        s_out.mkdir(parents=True, exist_ok=True)

        sample_q = chosen_q[sample]

        if verbose:
            print(f"  Processing sample: {sample} ({len(df_s):,} cells, q={sample_q})")

        before_dfs[sample] = df_s.copy()

        sweep_df = ly49c_sweep_metrics_one_sample(
            df_s=df_s,
            ct_col=ct_col,
            ly49c_col=ly49c_col,
            markers=markers,
            thresholds=thresholds,
            min_size=min_size,
            entropy_alpha=entropy_alpha_for_sweep,
            debug_probs=debug_probs,
        )
        sweep_csv = s_out / f"ly49c_sweep_metrics_alpha{entropy_alpha_for_sweep}.csv"
        sweep_df.to_csv(sweep_csv, index=False)

        sweep_ent_long = ly49c_sweep_entropy_orders_one_sample(
            df_s=df_s,
            sample_name=str(sample),
            ct_col=ct_col,
            ly49c_col=ly49c_col,
            markers=markers,
            thresholds=thresholds,
            orders=orders,
            min_size=min_size,
            debug_probs=debug_probs,
        )
        sweep_entropy_all_samples.append(sweep_ent_long)

        sweep_metrics_png = s_out / "ly49c_sweep_metrics_2x3.png"
        sweep_ent_this_sample = sweep_ent_long[sweep_ent_long["sample"] == str(sample)]
        plot_sweep_metrics_2x3(
            sweep_df=sweep_df,
            sweep_entropy_df=sweep_ent_this_sample,
            out_png=sweep_metrics_png,
            title=f"{dataset_name} | {sample} | Ly49C sweep metrics",
            sample_q=sample_q,
        )

        df_after, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, sample_q)

        after_dfs[sample] = df_after.copy()

        raw_b, prop_b, n_b = clonotype_raw_and_prop_from_raw(df_s, ct_col, markers, min_size=min_size)
        raw_a, prop_a, n_a = clonotype_raw_and_prop_from_raw(df_after, ct_col, markers, min_size=min_size)

        pdf_out = s_out / f"clonotypes_MEAN_and_PROP_before_after_q{sample_q:.3f}.pdf"
        plot_raw_and_prop_before_after_4x8_pdf(
            raw_b, raw_a, prop_b, prop_a, n_b, n_a,
            peptides=peptides,
            out_pdf=pdf_out,
            title=f"{dataset_name} | {sample} | clonotypes≥{min_size} MEAN+PROP before/after (q={sample_q:.3f}, Ts={Ts:.2f})",
        )

        renyi_base = renyi_profile_all_clonotypes_ge5(
            df_s=df_s, ct_col=ct_col, markers=markers, min_size=min_size, orders=orders, debug=debug_probs
        )
        renyi_after = renyi_profile_all_clonotypes_ge5(
            df_s=df_after, ct_col=ct_col, markers=markers, min_size=min_size, orders=orders, debug=debug_probs
        )

        renyi_base_csv = s_out / f"renyi_all_ge{min_size}_baseline.csv"
        renyi_after_csv = s_out / f"renyi_all_ge{min_size}_filtered_q{sample_q:.3f}.csv"
        renyi_base.to_csv(renyi_base_csv, index=False)
        renyi_after.to_csv(renyi_after_csv, index=False)

        plot_renyi_spaghetti_all_clonotypes(
            renyi_base,
            out_png=s_out / "renyi_spaghetti_baseline.png",
            title=f"{dataset_name} | {sample} | Renyi spaghetti (ALL clonotypes≥{min_size}) | baseline",
            alpha_linewidth=1.8,
            alpha_opacity=0.6,
        )
        plot_renyi_spaghetti_all_clonotypes(
            renyi_after,
            out_png=s_out / f"renyi_spaghetti_filtered_q{sample_q:.3f}.png",
            title=f"{dataset_name} | {sample} | Renyi spaghetti (ALL clonotypes≥{min_size}) | filtered q={sample_q:.3f}",
            alpha_linewidth=1.8,
            alpha_opacity=0.6,
        )

        pairs = coherence_before_after_table(
            df_before=df_s,
            df_after=df_after,
            ct_col=ct_col,
            markers=markers,
            min_size=min_size,
            debug_probs=debug_probs,
        )
        coh_csv = s_out / f"coherence_before_vs_after_q{sample_q:.3f}.csv"
        coh_png = s_out / f"coherence_before_vs_after_q{sample_q:.3f}.png"
        pairs.to_csv(coh_csv, index=False)

        chosen_row = sweep_df[np.isclose(sweep_df["percentile"], sample_q)]
        if not chosen_row.empty:
            mean_coh_change = float(chosen_row["mean_coherence_change"].iloc[0])
        else:
            mean_coh_change = None

        plot_coherence_before_after_scatter(
            pairs,
            out_png=coh_png,
            title=f"{dataset_name} | {sample} | coherence before vs after (q={sample_q:.3f})",
            mean_coherence_change=mean_coh_change,
        )

        coh_by_clonotype_pdf = s_out / f"clonotypes_by_coherence_q{sample_q:.3f}.pdf"
        plot_all_clonotypes_by_coherence_pdf(
            df=df_after,
            ct_col=ct_col,
            markers=markers,
            peptides=peptides,
            out_pdf=coh_by_clonotype_pdf,
            title_prefix=f"{dataset_name} | {sample} | q={sample_q:.3f}",
            min_size=min_size,
            normalize=True,
            plots_per_page=12,
            debug_probs=debug_probs,
            rank_map=None
        )

        index_rows.append({
            "dataset": dataset_name,
            "sample": sample,
            "n_cells_before": int(len(df_s)),
            "n_cells_after": int(len(df_after)),
            "n_clonotypes_ge5_before": int(len(n_b)),
            "n_clonotypes_ge5_after": int(len(n_a)),
            "chosen_q": float(sample_q),
            "Ts": float(Ts),
            "sweep_csv": str(sweep_csv),
            "sweep_metrics_png": str(sweep_metrics_png),
            "before_after_pdf": str(pdf_out),
            "renyi_base_csv": str(renyi_base_csv),
            "renyi_after_csv": str(renyi_after_csv),
            "coherence_csv": str(coh_csv),
            "coherence_png": str(coh_png),
            "clonotypes_by_coherence_pdf": str(coh_by_clonotype_pdf)
        })

    if sweep_entropy_all_samples:
        sweep_all = pd.concat(sweep_entropy_all_samples, ignore_index=True)
        plot_entropy_sweep_overlay_alpha_by_sample_2x3(
            sweep_all,
            out_png=ds_out / "ly49c_entropy_sweep_overlay_alpha0to5_2x3.png",
            title=f"{dataset_name} | ΔHα(q) sweep overlay α=0..5 (first 6 samples)",
            chosen_q=chosen_q,
        )

    index_df = pd.DataFrame(index_rows)
    index_df.to_csv(ds_out / "INDEX.csv", index=False)
    
    return index_df, before_dfs, after_dfs

In [104]:
def classify_clonotype_specificity(
    X: np.ndarray,
    peptides: list[str],
    control_peptide: str,
    single_threshold: float = 0.85,
    min_prop_threshold: float = 0.05,
    percentile_threshold: float = 95.0,
    control_fold_threshold: float = 2.0,
) -> dict:
    """
    Classify a clonotype's specificity using percentile-based background threshold.
    
    Pipeline:
    1. Single-specific gate: If one peptide ≥ single_threshold of total counts → "single"
    2. Background: peptides with mean proportion < min_prop_threshold
    3. Threshold: percentile_threshold of pooled background COUNTS
    4. Above-background: candidates (≥5%) with mean count > threshold AND > 2× control
    5. Classification based on count of above-background peptides
    
    Parameters
    ----------
    X : np.ndarray
        Raw counts matrix, shape (n_cells, n_peptides)
    peptides : list[str]
        Peptide names corresponding to columns of X
    control_peptide : str
        Name of degenerate control peptide (e.g., "EEEPVKKI" or "SYFPEITHI")
    single_threshold : float
        Proportion threshold for immediate single classification (default 0.85)
    min_prop_threshold : float
        Minimum proportion to be considered a candidate (default 0.05)
    percentile_threshold : float
        Percentile of background counts for threshold (default 95.0)
    control_fold_threshold : float
        Must be > this × control mean to be above-background (default 2.0)
    
    Returns
    -------
    dict with keys:
        - classification: "single", "dual", or "multi-specific"
        - above_background_peptides: list of peptides above threshold
        - above_background_proportions: their proportions
        - dominant_peptide: peptide with highest mean count
        - dominant_proportion: its proportion
        - candidate_peptides: list of peptides ≥ min_prop_threshold
        - background_peptides: list of peptides < min_prop_threshold
        - background_threshold_count: the percentile-based threshold (in counts)
        - control_mean_count: mean count of control peptide
        - prop_vector: normalized proportions for all peptides
        - mean_counts: mean raw counts for all peptides
    """
    n_cells, n_peptides = X.shape
    mean_counts = X.mean(axis=0)
    total_mean = mean_counts.sum()
    prop_vector = mean_counts / total_mean if total_mean > 0 else np.zeros(n_peptides)
    
    if control_peptide in peptides:
        control_idx = peptides.index(control_peptide)
        control_mean_count = float(mean_counts[control_idx])
    else:
        raise ValueError(f"Control peptide '{control_peptide}' not found in peptides")
    
    result = {
        "prop_vector": prop_vector,
        "mean_counts": mean_counts,
        "control_mean_count": control_mean_count,
        "n_cells": n_cells,
    }
    
    # Single specificity
    max_prop = prop_vector.max()
    max_idx = int(prop_vector.argmax())
    
    if max_prop >= single_threshold:
        median_counts = np.median(X, axis=0)
        median_dominant_idx = int(median_counts.argmax())
        result["classification"] = "single"
        result["above_background_peptides"] = [peptides[max_idx]]
        result["above_background_proportions"] = [float(max_prop)]
        result["dominant_peptide"] = peptides[max_idx]
        result["dominant_proportion"] = float(max_prop)
        result["median_dominant_peptide"] = peptides[median_dominant_idx]
        result["median_dominant_proportion"] = float(median_counts[median_dominant_idx] / median_counts.sum() if median_counts.sum() > 0 else 0.0)
        result["candidate_peptides"] = [peptides[max_idx]]
        result["background_peptides"] = [p for i, p in enumerate(peptides) if i != max_idx]
        result["background_threshold_count"] = np.nan
        return result
    
    # Background counts
    background_mask = prop_vector < min_prop_threshold
    candidate_mask = prop_vector >= min_prop_threshold
    
    background_peptides = [p for p, is_bg in zip(peptides, background_mask) if is_bg]
    candidate_peptides = [p for p, is_cand in zip(peptides, candidate_mask) if is_cand]
    
    result["background_peptides"] = background_peptides
    result["candidate_peptides"] = candidate_peptides
    
    # Percentile
    if background_mask.sum() > 0:
        background_counts_pooled = X[:, background_mask].flatten()
        background_threshold = float(np.percentile(background_counts_pooled, percentile_threshold))
    else:
        background_threshold = 0.0
    
    result["background_threshold_count"] = background_threshold
    
    # Not background
    above_background_peptides = []
    above_background_proportions = []
    
    for i, (peptide, prop, mc) in enumerate(zip(peptides, prop_vector, mean_counts)):
        if candidate_mask[i]:
            is_above_threshold = mc > background_threshold
            is_above_control = mc > control_mean_count * control_fold_threshold
            
            if is_above_threshold and is_above_control:
                above_background_peptides.append(peptide)
                above_background_proportions.append(float(prop))
    
    result["above_background_peptides"] = above_background_peptides
    result["above_background_proportions"] = above_background_proportions
    
    # Dominant peptide
    dominant_idx = int(mean_counts.argmax())
    result["dominant_peptide"] = peptides[dominant_idx]
    result["dominant_proportion"] = float(prop_vector[dominant_idx])

    median_counts = np.median(X, axis=0)
    median_dominant_idx = int(median_counts.argmax())
    result["median_dominant_peptide"] = peptides[median_dominant_idx]
    result["median_dominant_proportion"] = float(prop_vector[median_dominant_idx]/median_counts.sum() if median_counts.sum() > 0 else 0.0)
    
    n_above = len(above_background_peptides)
    
    if n_above <= 1:
        result["classification"] = "single"
    elif n_above == 2:
        result["classification"] = "dual"
    else:
        result["classification"] = "multi-specific"
    
    return result

In [105]:
def classify_all_clonotypes(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    control_peptide: str,
    min_size: int = 5,
    single_threshold: float = 0.85,
    min_prop_threshold: float = 0.05,
    percentile_threshold: float = 95.0,
    control_fold_threshold: float = 2.0,
) -> pd.DataFrame:
    """
    Classify all clonotypes in a dataset using the percentile-based pipeline.
    
    Parameters
    ----------
    df : pd.DataFrame
        Cell-level data
    ct_col : str
        Clonotype column name
    markers : list[str]
        Dextramer marker columns
    peptides : list[str]
        Short peptide names
    control_peptide : str
        Name of control peptide (e.g., "EEEPVKKI" or "SYFPEITHI")
    min_size : int
        Minimum cells per clonotype (default 5)
    single_threshold : float
        Proportion for single-specific gate (default 0.85)
    min_prop_threshold : float
        Proportion threshold for background (default 0.05)
    percentile_threshold : float
        Percentile for background threshold (default 95.0)
    control_fold_threshold : float
        Fold over control required (default 2.0)
    
    Returns
    -------
    pd.DataFrame with classification results
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    
    rows = []
    
    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float)
        
        result = classify_clonotype_specificity(
            X=X,
            peptides=peptides,
            control_peptide=control_peptide,
            single_threshold=single_threshold,
            min_prop_threshold=min_prop_threshold,
            percentile_threshold=percentile_threshold,
            control_fold_threshold=control_fold_threshold,
        )
        
        rows.append({
            "clonotype": ct,
            "n_cells": int(len(g)),
            "classification": result["classification"],
            "above_background_peptides": result["above_background_peptides"],
            "n_above_background": len(result["above_background_peptides"]),
            "above_background_proportions": result["above_background_proportions"],
            "dominant_peptide": result["dominant_peptide"],
            "dominant_proportion": result["dominant_proportion"],
            "median_dominant_peptide": result["median_dominant_peptide"],
            "median_dominant_proportion": result["median_dominant_proportion"],
            "dominant_peptide_agrees": result["dominant_peptide"] == result["median_dominant_peptide"],
            "candidate_peptides": result["candidate_peptides"],
            "n_candidates": len(result["candidate_peptides"]),
            "background_peptides": result["background_peptides"],
            "background_threshold_count": result["background_threshold_count"],
            "control_mean_count": result["control_mean_count"],
            "prop_vector": result["prop_vector"],
            "mean_counts": result["mean_counts"],
        })
    
    return pd.DataFrame(rows)

In [106]:
def leave_one_out_coherence(
    P: np.ndarray,
    markers: list[str],
) -> dict[str, float]:
    """
    Compute coherence after removing each peptide/marker in turn.
    
    Parameters
    ----------
    P : np.ndarray
        Row-normalized binding matrix (n_cells x n_peptides)
    markers : list[str]
        Marker/peptide names
    
    Returns
    -------
    dict mapping marker -> coherence when that marker is excluded
    """
    n_cells, n_markers = P.shape
    
    if n_cells < 2:
        return {m: np.nan for m in markers}
    
    results = {}
    
    for i, marker in enumerate(markers):
        P_reduced = np.delete(P, i, axis=1)
        
        row_sums = P_reduced.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        P_renorm = P_reduced / row_sums
        
        coh = mean_pairwise_cosine_similarity(P_renorm)
        results[marker] = coh
    
    return results

In [107]:
def compute_clonotype_coherence_analysis(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    min_size: int = 5,
    min_prop_for_ranking: float = 0.05
) -> pd.DataFrame:
    """
    For each clonotype, compute:
      - Full coherence
      - Leave-one-out coherence for each peptide
      - Delta coherence (LOO - full) for each peptide
    
    Parameters
    ----------
    df : pd.DataFrame
        Data with cells as rows
    ct_col : str
        Clonotype column name
    markers : list[str]
        Dextramer marker columns
    peptides : list[str]
        Short peptide names (for output column names)
    min_size : int
        Minimum cells per clonotype
    min_prop_for_ranking : float
        Only consider peptides with >= this proportion when determining
        most_disruptive / most_stabilizing. Default 0.05 (5%).
    
    Returns
    -------
    pd.DataFrame with columns:
        clonotype, n_cells, coherence_full,
        coh_without_{peptide} for each peptide,
        delta_coh_{peptide} for each peptide,
        most_disruptive_peptide (removing it raises coherence most),
        most_stabilizing_peptide (removing it drops coherence most)
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    
    rows = []
    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float)
        P = row_normalise(X)
        
        mean_counts = X.mean(axis=0)
        total = mean_counts.sum()
        prop_vector = mean_counts / total if total > 0 else np.zeros_like(mean_counts)
        
        coh_full = mean_pairwise_cosine_similarity(P)
        
        loo_results = leave_one_out_coherence(P, markers)
        
        row = {
            "clonotype": ct,
            "n_cells": int(len(g)),
            "coherence_full": coh_full,
        }
        
        deltas = {}
        for i, (marker, peptide) in enumerate(zip(markers, peptides)):
            coh_without = loo_results[marker]
            delta = coh_without - coh_full if np.isfinite(coh_without) and np.isfinite(coh_full) else np.nan
            
            row[f"coh_without_{peptide}"] = coh_without
            row[f"delta_coh_{peptide}"] = delta
            row[f"prop_{peptide}"] = prop_vector[i]
            
            if prop_vector[i] >= min_prop_for_ranking and np.isfinite(delta):
                deltas[peptide] = delta
        
        if deltas:
            row["most_disruptive_peptide"] = max(deltas, key=deltas.get)
            row["most_disruptive_delta"] = max(deltas.values())
            row["most_stabilizing_peptide"] = min(deltas, key=deltas.get)
            row["most_stabilizing_delta"] = min(deltas.values())
        else:
            row["most_disruptive_peptide"] = None
            row["most_disruptive_delta"] = np.nan
            row["most_stabilizing_peptide"] = None
            row["most_stabilizing_delta"] = np.nan
        
        rows.append(row)
    
    return pd.DataFrame(rows)

In [108]:
def full_specificity_analysis(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    control_peptide: str,
    min_size: int = 5,
    single_threshold: float = 0.85,
    min_prop_threshold: float = 0.05,
    percentile_threshold: float = 95.0,
    control_fold_threshold: float = 2.0,
    min_prop_for_coherence_ranking: float = 0.05,
) -> pd.DataFrame:
    spec_df = classify_all_clonotypes(
        df=df, ct_col=ct_col, markers=markers, peptides=peptides,
        control_peptide=control_peptide, min_size=min_size,
        single_threshold=single_threshold, min_prop_threshold=min_prop_threshold,
        percentile_threshold=percentile_threshold, control_fold_threshold=control_fold_threshold,
    )
    coh_df = compute_clonotype_coherence_analysis(
        df=df, ct_col=ct_col, markers=markers, peptides=peptides,
        min_size=min_size, min_prop_for_ranking=min_prop_for_coherence_ranking,
    )
    return spec_df.merge(coh_df, on=["clonotype", "n_cells"], how="outer")

In [109]:
def plot_clonotypes_with_specificity_highlights_pdf(
    df: pd.DataFrame,
    spec_df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    out_pdf: Path,
    title_prefix: str = "",
    min_size: int = 5,
    normalize: bool = True,
    jitter_width: float = 0.25,
    plots_per_page: int = 12,
    sample_col: str | None = None,
    min_prop_threshold: float = 0.05,
):
    """
    Multi-page PDF showing per-cell binding distributions for each clonotype,
    with above-background peptides highlighted.
    
    Parameters
    ----------
    df : pd.DataFrame
        Cell-level data
    spec_df : pd.DataFrame
        Output from full_specificity_analysis() or classify_all_clonotypes()
    ct_col : str
        Clonotype column name
    markers : list[str]
        Dextramer marker columns
    peptides : list[str]
        Short peptide names for x-axis labels
    out_pdf : Path
        Output PDF path
    title_prefix : str
        Prefix for page titles
    min_size : int
        Minimum cells per clonotype
    normalize : bool
        If True, plot proportions. If False, plot raw counts.
    jitter_width : float
        Horizontal jitter for dots
    plots_per_page : int
        Number of panels per page
    sample_col : str | None
        If provided, color points by sample origin
    min_prop_threshold : float
        Proportion threshold for drawing reference line (default 0.05)
    """
    out_pdf = Path(out_pdf)
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    
    class_order = {"single": 0, "dual": 1, "multi-specific": 2}
    spec_df = spec_df.copy()
    spec_df["_class_order"] = spec_df["classification"].map(class_order).fillna(3)
    
    if "coherence_full" in spec_df.columns:
        spec_df = spec_df.sort_values(
            ["_class_order", "coherence_full"],
            ascending=[True, False]
        ).reset_index(drop=True)
    else:
        spec_df = spec_df.sort_values(
            ["_class_order", "n_cells"],
            ascending=[True, False]
        ).reset_index(drop=True)
    
    n_clonotypes = len(spec_df)
    print(f"Plotting {n_clonotypes} clonotypes with specificity highlights")
    
    if sample_col is not None and sample_col in df.columns:
        unique_samples = sorted(df[sample_col].unique())
        sample_cmap = plt.cm.tab10
        sample_colors = {s: sample_cmap(i % 10) for i, s in enumerate(unique_samples)}
        color_by_sample = True
    else:
        color_by_sample = False
    
    if plots_per_page == 12:
        nrows, ncols = 3, 4
    elif plots_per_page == 16:
        nrows, ncols = 4, 4
    elif plots_per_page == 9:
        nrows, ncols = 3, 3
    else:
        ncols = 4
        nrows = (plots_per_page + ncols - 1) // ncols
    
    n_pages = (n_clonotypes + plots_per_page - 1) // plots_per_page
    
    ylabel = "Proportion" if normalize else "Raw count"
    xs = np.arange(len(peptides))
    
    above_bg_color = "darkred"
    candidate_color = "darkorange"
    background_color = "steelblue"
    
    title_colors = {"single": "darkgreen", "dual": "darkorange", "multi-specific": "darkred"}
    
    with PdfPages(out_pdf) as pdf:
        for page_idx in range(n_pages):
            start = page_idx * plots_per_page
            end = min(start + plots_per_page, n_clonotypes)
            chunk = spec_df.iloc[start:end]
            
            fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
            axes = axes.flatten()
            
            for i, (_, row) in enumerate(chunk.iterrows()):
                ax = axes[i]
                ct = row["clonotype"]
                classification = row["classification"]
                above_bg = row["above_background_peptides"]
                candidates = row["candidate_peptides"]
                dominant = row["dominant_peptide"]
                n_cells = row["n_cells"]
                bg_threshold = row["background_threshold_count"]
                
                coherence = row.get("coherence_full", np.nan)
                
                df_ct = df[df[ct_col] == ct]
                X = df_ct[markers].to_numpy(dtype=float)
                
                mean_counts = X.mean(axis=0)
                total = mean_counts.sum()
                mean_props = mean_counts / total if total > 0 else np.zeros_like(mean_counts)
                
                if normalize:
                    row_sums = X.sum(axis=1, keepdims=True)
                    row_sums[row_sums == 0] = 1.0
                    X_plot = X / row_sums
                else:
                    X_plot = X
                
                peptide_status = []
                for p in peptides:
                    if p in above_bg:
                        peptide_status.append("above")
                    elif p in candidates:
                        peptide_status.append("candidate")
                    else:
                        peptide_status.append("background")
                
                if color_by_sample:
                    samples_ct = df_ct[sample_col].to_numpy()
                    sample_counts = {}
                    
                    for j in range(X_plot.shape[0]):
                        jitter = np.random.uniform(-jitter_width, jitter_width, len(peptides))
                        sample_name = samples_ct[j]
                        color = sample_colors[sample_name]
                        
                        for k in range(len(peptides)):
                            if peptide_status[k] == "above":
                                alpha_val, size = 0.8, 25
                            elif peptide_status[k] == "candidate":
                                alpha_val, size = 0.6, 18
                            else:
                                alpha_val, size = 0.35, 12
                            
                            ax.scatter(xs[k] + jitter[k], X_plot[j, k], alpha=alpha_val, s=size, c=[color], edgecolors="none")
                        
                        sample_counts[sample_name] = sample_counts.get(sample_name, 0) + 1
                else:
                    for j in range(X_plot.shape[0]):
                        jitter = np.random.uniform(-jitter_width, jitter_width, len(peptides))
                        for k in range(len(peptides)):
                            if peptide_status[k] == "above":
                                c, alpha, s = above_bg_color, 0.6, 18
                            elif peptide_status[k] == "candidate":
                                c, alpha, s = candidate_color, 0.5, 14
                            else:
                                c, alpha, s = background_color, 0.4, 10
                            ax.scatter(xs[k] + jitter[k], X_plot[j, k], alpha=alpha, s=s, c=c, edgecolors="none")
                
                if normalize:
                    median_vals = np.median(X_plot, axis=0)
                else:
                    median_vals = np.median(X, axis=0)
                
                for k in range(len(peptides)):
                    if peptide_status[k] == "above":
                        marker_color = "darkred"
                    elif peptide_status[k] == "candidate":
                        marker_color = "darkorange"
                    else:
                        marker_color = "black"
                    
                    if normalize:
                        ax.scatter(xs[k], mean_props[k], marker="D", s=50, color=marker_color, zorder=10)  # mean
                        ax.scatter(xs[k], median_vals[k], marker="_", s=100, color=marker_color, zorder=10, linewidths=2)  # median
                    else:
                        ax.scatter(xs[k], mean_counts[k], marker="D", s=50, color=marker_color, zorder=10)  # mean
                        ax.scatter(xs[k], median_vals[k], marker="_", s=100, color=marker_color, zorder=10, linewidths=2)  # median
                
                if normalize:
                    ax.axhline(min_prop_threshold, color="red", ls="--", lw=1.0, alpha=0.7, zorder=5)
                
                ax.set_xticks(xs)
                xticklabels = ax.set_xticklabels(peptides, rotation=90, fontsize=6)
                for k, label in enumerate(xticklabels):
                    if peptide_status[k] == "above":
                        label.set_color("darkred")
                        label.set_fontweight("bold")
                    elif peptide_status[k] == "candidate":
                        label.set_color("darkorange")
                
                ax.tick_params(axis="y", labelsize=7)
                ax.set_ylabel(ylabel, fontsize=7)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                
                # Use pre-existing rank if available, otherwise compute from position
                if "rank" in row.index:
                    rank = int(row["rank"])
                else:
                    rank = start + i + 1
                above_str = "+".join(above_bg[:2]) if above_bg else "none"
                if len(above_bg) > 2:
                    above_str += f"+{len(above_bg)-2}more"
                
                coh_str = f"coh={coherence:.3f}" if np.isfinite(coherence) else ""
                thresh_str = f"thresh={bg_threshold:.1f}" if np.isfinite(bg_threshold) else ""
                
                if color_by_sample:
                    sample_str = ", ".join([f"{s.split('_')[-1]}:{c}" for s, c in sorted(sample_counts.items())])
                    ax.set_title(
                        f"#{rank} {ct}\n{classification.upper()} | Above BG: {above_str}\n"
                        f"n={n_cells} [{sample_str}] {thresh_str}",
                        fontsize=6,
                        color=title_colors.get(classification, "black")
                    )
                else:
                    ax.set_title(
                        f"#{rank} {ct}\n{classification.upper()} | Above BG: {above_str}\n"
                        f"n={n_cells} | {coh_str} {thresh_str}",
                        fontsize=6,
                        color=title_colors.get(classification, "black")
                    )
            
            for j in range(len(chunk), len(axes)):
                axes[j].axis("off")
            
            if page_idx == 0:
                if color_by_sample:
                    legend_handles = [
                        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=sample_colors[s], markersize=8, label=s.split('_')[-1] if '_' in s else s)
                        for s in unique_samples
                    ]
                    legend_title = "Sample"
                else:
                    legend_handles = [
                        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=above_bg_color, markersize=8, label='Above BG (≥5%, >95th pctl, >2×ctrl)'),
                        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=candidate_color, markersize=8, label='Candidate (≥5%)'),
                        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=background_color, markersize=8, label='Background (<5%)'),
                        plt.Line2D([0], [0], ls='--', color='red', alpha=0.7, label='5% threshold'),
                    ]
                    legend_title = "Legend"
                
                fig.legend(
                    handles=legend_handles,
                    loc='upper right',
                    bbox_to_anchor=(0.99, 0.99),
                    fontsize=7,
                    frameon=True,
                    title=legend_title
                )
            
            fig.suptitle(
                f"{title_prefix} | Clonotypes by specificity (page {page_idx + 1}/{n_pages})",
                fontsize=12, y=0.995
            )
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pdf.savefig(fig)
            plt.close(fig)
    
    print(f"Saved: {out_pdf}")

In [110]:
def plot_correlation_analysis_pdf(
    df: pd.DataFrame,
    spec_df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    out_pdf: Path,
    title_prefix: str = "",
):
    out_pdf = Path(out_pdf)
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    spec_df = spec_df.copy()
    if "rank" in spec_df.columns:
        spec_df = spec_df.sort_values("rank", ascending=True).reset_index(drop=True)
    else:
        class_order = {"single": 0, "dual": 1, "multi-specific": 2}
        spec_df["_class_order"] = spec_df["classification"].map(class_order).fillna(3)
        spec_df = spec_df.sort_values(["_class_order", "n_cells"], ascending=[True, False]).reset_index(drop=True)
    n_clonotypes = len(spec_df)
    print(f"Plotting correlation analysis for {n_clonotypes} clonotypes")

    ncols = 4
    nrows = (len(peptides) + ncols - 1) // ncols

    def _plot_regression_page(pdf, X, dominant, dominant_prop, dominant_label, peptides, above_bg, ct, classification, n_cells, title_prefix, nrows, ncols, rank):
        dominant_counts = X[:, peptides.index(dominant)]
        fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
        axes = axes.flatten()
        for i, peptide in enumerate(peptides):
            ax = axes[i]
            other_counts = X[:, i]

            if peptide == dominant: color, marker = "purple",    "s"
            elif peptide in above_bg: color, marker = "darkred",   "o"
            else: color, marker = "steelblue", "o"

            ax.scatter(dominant_counts, other_counts, c=color, alpha=0.6, s=25, marker=marker, edgecolors="white", linewidth=0.5)
            
            if np.std(dominant_counts) > 0 and np.std(other_counts) > 0:
                r, p = stats.pearsonr(dominant_counts, other_counts)
                slope, intercept = np.polyfit(dominant_counts, other_counts, 1)
                x_line = np.array([dominant_counts.min(), dominant_counts.max()])
                ax.plot(x_line, slope * x_line + intercept, 'k--', alpha=0.5, lw=1)
                ax.set_title(f"{peptide}\nr={r:.3f}, p={p:.2e}", fontsize=8, color="darkred" if peptide in above_bg else "black")
            else:
                ax.set_title(f"{peptide}", fontsize=8)
            ax.set_xlabel(f"{dominant} counts ({dominant_label})", fontsize=7)
            ax.set_ylabel(f"{peptide} counts", fontsize=7)
            ax.tick_params(labelsize=6)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(len(peptides), len(axes)):
            axes[j].axis("off")

        above_bg_str = ", ".join(above_bg) if above_bg else "none"
        fig.suptitle(
            f"{title_prefix} | #{rank} {ct}\n"
            f"Classification: {classification.upper()} | n_cells={n_cells}\n"
            f"Dominant ({dominant_label}): {dominant} ({dominant_prop:.1%}) | Above background: {above_bg_str}",
            fontsize=10, y=0.98
        )
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        pdf.savefig(fig)
        plt.close(fig)

    with PdfPages(out_pdf) as pdf:
        for idx, row in spec_df.iterrows():
            ct = row["clonotype"]
            classification = row["classification"]
            above_bg = row["above_background_peptides"]
            dominant_mean = row["dominant_peptide"]
            dominant_mean_prop = row["dominant_proportion"]
            dominant_median = row["median_dominant_peptide"]
            dominant_median_prop = row["median_dominant_proportion"]
            agrees = row["dominant_peptide_agrees"]
            n_cells = row["n_cells"]

            g = df[df[ct_col] == ct]
            X = g[markers].to_numpy(dtype=float)

            rank = int(row["rank"]) if "rank" in row.index else idx + 1

            _plot_regression_page(pdf, X, dominant_mean, dominant_mean_prop, "mean", peptides, above_bg, ct, classification, n_cells, title_prefix, nrows, ncols, rank)

            # Only plot median-dominant page if it disagrees with mean
            if not agrees:
                _plot_regression_page(pdf, X, dominant_median, dominant_median_prop, "median", peptides, above_bg, ct, classification, n_cells, title_prefix, nrows, ncols, rank)

    print(f"Saved: {out_pdf}")

In [111]:
def plot_static_group_metrics_2x3(
    group_dfs: dict[str, pd.DataFrame],
    ct_col: str,
    markers: list[str],
    peptides: list[str],
    out_png: Path,
    title: str,
    min_size: int = 5,
    renyi_orders: list[float] | None = None,
    debug_probs: bool = False,
):
    """
    Static 2x3 comparison across named groups (no sweep).
    
    Parameters
    ----------
    group_dfs : dict[str, pd.DataFrame]
        Ordered dict of group_name -> dataframe
    """
    if renyi_orders is None:
        renyi_orders = [0, 1, 2, 3, 4, 5]

    group_names = list(group_dfs.keys())
    colors = plt.cm.tab10(np.linspace(0, 0.5, len(group_names)))

    records = []
    for gname, df_g in group_dfs.items():
        summaries = compute_clonotype_summaries(
            df_g, ct_col, markers, min_size=min_size,
            entropy_alpha=1.0, debug_probs=debug_probs
        )
        if not summaries:
            continue

        coh_vals = [s["mean_coherence"] for s in summaries.values() if np.isfinite(s["mean_coherence"])]

        patterns = np.array([s["mean_pattern"] for s in summaries.values()])
        if len(patterns) >= 2:
            cos_dists = pdist(patterns, metric="cosine")
            l1_dists  = pdist(patterns, metric="cityblock")
            mean_cos  = float(np.mean(cos_dists))
            mean_l1   = float(np.mean(l1_dists))
        else:
            mean_cos = np.nan
            mean_l1  = np.nan

        renyi_by_order = {}
        for a in renyi_orders:
            ent_vals = []
            for ct, g in df_g.groupby(ct_col):
                if ct not in summaries:
                    continue
                X = g[markers].to_numpy(dtype=float)
                P = row_normalise(X, debug=debug_probs)
                ent_vals.append(float(np.mean(renyi_entropy(P, alpha=float(a), axis=1))))
            renyi_by_order[a] = float(np.mean(ent_vals)) if ent_vals else np.nan

        records.append({
            "group": gname,
            "mean_coherence": float(np.mean(coh_vals)) if coh_vals else np.nan,
            "sem_coherence":  float(np.std(coh_vals) / np.sqrt(len(coh_vals))) if len(coh_vals) > 1 else 0.0,
            "mean_cosine_dist": mean_cos,
            "mean_l1_dist": mean_l1,
            "n_clonotypes": len(summaries),
            "renyi": renyi_by_order,
        })

    if not records:
        print("No data to plot")
        return

    xs = np.arange(len(records))
    group_labels = [r["group"] for r in records]

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))

    ax = axes[0, 0]
    vals = [r["mean_coherence"] for r in records]
    errs = [r["sem_coherence"]  for r in records]
    ax.bar(xs, vals, yerr=errs, color=colors, alpha=0.85, capsize=4)
    ax.set_xticks(xs); ax.set_xticklabels(group_labels, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("Mean coherence")
    ax.set_title("Coherence")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[0, 1]
    vals = [r["mean_cosine_dist"] for r in records]
    ax.bar(xs, vals, color=colors, alpha=0.85)
    ax.set_xticks(xs); ax.set_xticklabels(group_labels, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("Mean pairwise cosine dist")
    ax.set_title("Pattern diversity (cosine)")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[0, 2]
    vals = [r["mean_l1_dist"] for r in records]
    ax.bar(xs, vals, color=colors, alpha=0.85)
    ax.set_xticks(xs); ax.set_xticklabels(group_labels, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("Mean pairwise L1 dist")
    ax.set_title("Pattern diversity (L1)")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[1, 0]
    for r, color in zip(records, colors):
        alphas = sorted(r["renyi"].keys())
        ents   = [r["renyi"][a] for a in alphas]
        ax.plot(alphas, ents, marker="o", lw=1.8, color=color, label=r["group"])
    ax.set_xlabel("Renyi order α")
    ax.set_ylabel("Mean Renyi entropy (bits)")
    ax.set_title("Renyi entropy profile")
    ax.legend(frameon=False, fontsize=8)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[1, 1]
    vals = [r["n_clonotypes"] for r in records]
    ax.bar(xs, vals, color=colors, alpha=0.85)
    ax.set_xticks(xs); ax.set_xticklabels(group_labels, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel(f"N clonotypes ≥{min_size} cells")
    ax.set_title("Clonotype count")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    ax = axes[1, 2]
    n_orders = len(renyi_orders)
    bar_width = 0.8 / len(records)
    for j, (r, color) in enumerate(zip(records, colors)):
        offsets = np.arange(n_orders) + j * bar_width - (len(records) - 1) * bar_width / 2
        vals = [r["renyi"][a] for a in renyi_orders]
        ax.bar(offsets, vals, bar_width, color=color, alpha=0.85, label=r["group"])
    ax.set_xticks(np.arange(n_orders))
    ax.set_xticklabels([f"α={a}" for a in renyi_orders], fontsize=8)
    ax.set_ylabel("Mean Renyi entropy (bits)")
    ax.set_title("Renyi entropy by order")
    ax.legend(frameon=False, fontsize=7)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    fig.suptitle(title, fontsize=12, y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_png}")

In [112]:
df_b10br = pd.read_csv(
    "../Data/20260116 Comparison 3/20251223 BL6-B10BR HTx HIL Clonotypes with ADT Counts.csv",
    index_col=0
)

df_balbc = pd.read_csv(
    "../Data/20260116 Comparison 3/20251218 BL6-BALBc HTx HIL Clonotypes with ADT Counts.csv",
    index_col=0
)

df_d7abc = pd.read_csv(
    "../Data/20260116 Comparison 3/20260114 B10BR D7ABC Kb LL Repertoire + ADT counts.csv"
)

public_clonotypes_df = pd.read_csv("../Data/Comp 3 - Public clonotypes (HIL and LL).csv")
public_clonotypes = set(public_clonotypes_df["Clonotpe"].tolist())
print(f"Public clonotypes: {len(public_clonotypes)}")

Public clonotypes: 64


In [113]:
chosen_q_b10br = {
    "BL6-B10BR_HTxA": 0.975,
    "BL6-B10BR_HTxB": 0.975,
    "BL6-B10BR_HTxC": 0.925,
}

chosen_q_balbc = {
    "BL6-BALBc_HTxA": 1.000,
    "BL6-BALBc_HTxB": 0.975,
    "BL6-BALBc_HTxC": 0.95,
}

chosen_q_syfpeithi = {
    "BL6-BALBc_HTxA": 1.000,
    "BL6-BALBc_HTxB": 0.920,
    "BL6-BALBc_HTxC": 0.980,
}

chosen_q_eeepvkki_d7abc = {
    "D7A": 0.85,
    "D7B": 0.90,
    "D7C": 1.00,
}

class_order = {"single": 0, "dual": 1, "multi-specific": 2}

In [ ]:
index_b10br, before_b10br, after_b10br = run_samplewise_pipeline(
    df=df_b10br,
    dataset_name="B10BR_HIL",
    markers=markers_b10br,
    peptides=peptides_b10br,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=chosen_q_b10br,
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5],
    debug_probs=False,
)

pairs_b10br_combined = plot_combined_coherence(
    before_dfs=before_b10br,
    after_dfs=after_b10br,
    samples_to_combine=["BL6-B10BR_HTxA", "BL6-B10BR_HTxB", "BL6-B10BR_HTxC"],
    markers=markers_b10br,
    output_path=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/coherence_combined_all"),
    title="B10BR_HIL | Combined (HTxA + HTxB + HTxC)",
    min_size=5,
)

df_b10br_combined = pd.concat([after_b10br[s] for s in ["BL6-B10BR_HTxA", "BL6-B10BR_HTxB", "BL6-B10BR_HTxC"]], ignore_index=True)

specificity_b10br = full_specificity_analysis(
    df=df_b10br_combined,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    control_peptide="EEEPVKKI",
    min_size=5,
    single_threshold=0.85,
    min_prop_threshold=0.05,
    percentile_threshold=0.95,
    control_fold_threshold=2.0
)

specificity_b10br["_class_order"] = specificity_b10br["classification"].map(class_order)
specificity_b10br = specificity_b10br.sort_values(["_class_order", "coherence_full"], ascending=[True, False]).reset_index(drop=True)
specificity_b10br["rank"] = specificity_b10br.index + 1
specificity_b10br = specificity_b10br.drop(columns=["_class_order"])

cols_b10br = ["rank", "clonotype"] + [c for c in specificity_b10br.columns if c not in ["rank", "clonotype", "prop_vector"]]

specificity_b10br[cols_b10br].to_csv(Path("Comparison3_Samplewise_Outputs/B10BR_HIL/specificity_analysis.csv"), index=False)

rank_map_b10br = dict(zip(specificity_b10br["clonotype"], specificity_b10br["rank"]))

plot_all_clonotypes_by_coherence_pdf(
    df=df_b10br_combined, ct_col=CT_COL, markers=markers_b10br, peptides=peptides_b10br,
    out_pdf=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/clonotypes_by_coherence_combined_ranked.pdf"),
    title_prefix="B10BR_HIL Combined | ranked by specificity order",
    min_size=5, normalize=True, plots_per_page=12,
    rank_map=rank_map_b10br,
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_b10br_combined,
    spec_df=specificity_b10br,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    out_pdf=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/counts_clonotypes_specificity_highlights.pdf"),
    title_prefix="B10BR_HIL Combined",
    normalize=False,
    sample_col=SAMPLE_COL,
    min_prop_threshold=0.05
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_b10br_combined,
    spec_df=specificity_b10br,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    out_pdf=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/prop_clonotypes_specificity_highlights.pdf"),
    title_prefix="B10BR_HIL Combined",
    normalize=True,
    sample_col=SAMPLE_COL,
    min_prop_threshold=0.05
)

plot_correlation_analysis_pdf(
    df=df_b10br_combined,
    spec_df=specificity_b10br,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    out_pdf=Path("Comparison3_Samplewise_Outputs/B10BR_HIL/specificity_correlations.pdf"),
    title_prefix="B10BR_HIL Combined",
)

In [ ]:
index_balbc, before_balbc, after_balbc = run_samplewise_pipeline(
    df=df_balbc,
    dataset_name="BALBc_HIL",
    markers=markers_balbc,
    peptides=peptides_balbc,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=chosen_q_balbc,
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5],
    debug_probs=False,
)

pairs_balbc_combined = plot_combined_coherence(
    before_dfs=before_balbc,
    after_dfs=after_balbc,
    samples_to_combine=["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"],
    markers=markers_balbc,
    output_path=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/coherence_combined_HTxB_HTxC"),
    title="BALBc_HIL | Combined (HTxB + HTxC)",
    min_size=5,
)

for sample in ["BL6-BALBc_HTxA", "BL6-BALBc_HTxB", "BL6-BALBc_HTxC"]:
    df_sample = after_balbc[sample]
    sweep_metrics = ly49c_sweep_metrics_one_sample(
        df_s=df_sample, ct_col=CT_COL, ly49c_col=SYFPEITHI_COL,
        markers=markers_balbc, thresholds=thresholds, min_size=5, entropy_alpha=1.0,
    )
    sweep_entropy = ly49c_sweep_entropy_orders_one_sample(
        df_s=df_sample, sample_name=sample, ct_col=CT_COL, ly49c_col=SYFPEITHI_COL,
        markers=markers_balbc, thresholds=thresholds, orders=[0, 1, 2, 3, 4, 5], min_size=5,
    )
    s_out = Path(f"Comparison3_Samplewise_Outputs/BALBc_HIL/{sample}")
    s_out.mkdir(parents=True, exist_ok=True)
    sweep_metrics.to_csv(s_out / "SYFPEITHI_sweep_metrics.csv", index=False)
    sweep_entropy.to_csv(s_out / "SYFPEITHI_sweep_entropy.csv", index=False)
    plot_sweep_metrics_2x3(
        sweep_df=sweep_metrics,
        sweep_entropy_df=sweep_entropy[sweep_entropy["sample"] == sample],
        out_png=s_out / "SYFPEITHI_sweep_metrics_2x3.png",
        title=f"BALBc HIL {sample} (post-Ly49C) | SYFPEITHI Sweep",
        sample_q=chosen_q_syfpeithi[sample],
    )

after_balbc_syfpeithi = {}
for sample, df_s in after_balbc.items():
    q = chosen_q_syfpeithi[sample]
    df_f, Ts = apply_ly49c_filter_one_sample(df_s, SYFPEITHI_COL, q)
    after_balbc_syfpeithi[sample] = df_f
    print(f"  {sample}: SYFPEITHI q={q} → {len(df_s):,} → {len(df_f):,} cells (Ts={Ts:.2f})")


pairs_balbc_combined = plot_combined_coherence(
    before_dfs=after_balbc,
    after_dfs=after_balbc_syfpeithi,
    samples_to_combine=["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"],
    markers=markers_balbc,
    output_path=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/coherence_combined_HTxB_HTxC_postSYFPEITHI"),
    title="BALBc_HIL | Combined (HTxB + HTxC) | post-Ly49C + SYFPEITHI",
    min_size=5,
)

df_balbc_combined = pd.concat([after_balbc_syfpeithi[s] for s in ["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"]], ignore_index=True)

df_balbc_unfiltered_combined = pd.concat(
    [before_balbc[s] for s in ["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"]], ignore_index=True
)
df_balbc_ly49c_combined = pd.concat(
    [after_balbc[s] for s in ["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"]], ignore_index=True
)

pairs_unfiltered_to_ly49c = plot_combined_coherence(
    before_dfs=before_balbc,
    after_dfs=after_balbc,
    samples_to_combine=["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"],
    markers=markers_balbc,
    output_path=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/coherence_unfiltered_to_ly49c"),
    title="BALBc_HIL | HTxB + HTxC | Unfiltered → Ly49C",
    min_size=5,
)

pairs_ly49c_to_syfpeithi = plot_combined_coherence(
    before_dfs=after_balbc,
    after_dfs=after_balbc_syfpeithi,
    samples_to_combine=["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"],
    markers=markers_balbc,
    output_path=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/coherence_ly49c_to_syfpeithi"),
    title="BALBc_HIL | HTxB + HTxC | Ly49C → SYFPEITHI",
    min_size=5,
)

pairs_unfiltered_to_fully_filtered = plot_combined_coherence(
    before_dfs=before_balbc,
    after_dfs=after_balbc_syfpeithi,
    samples_to_combine=["BL6-BALBc_HTxB", "BL6-BALBc_HTxC"],
    markers=markers_balbc,
    output_path=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/coherence_unfiltered_to_fully_filtered"),
    title="BALBc_HIL | HTxB + HTxC | Unfiltered → Ly49C + SYFPEITHI",
    min_size=5,
)

specificity_balbc = full_specificity_analysis(
    df=df_balbc_combined,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    control_peptide="SYFPEITHI",
    min_size=5,
    single_threshold=0.85,
    min_prop_threshold=0.05,
    percentile_threshold=0.95,
    control_fold_threshold=2.0
)

specificity_balbc["_class_order"] = specificity_balbc["classification"].map(class_order)
specificity_balbc = specificity_balbc.sort_values(["_class_order", "coherence_full"], ascending=[True, False]).reset_index(drop=True)
specificity_balbc["rank"] = specificity_balbc.index + 1
specificity_balbc = specificity_balbc.drop(columns=["_class_order"])

cols_balbc = ["rank", "clonotype"] + [c for c in specificity_balbc.columns if c not in ["rank", "clonotype", "prop_vector"]]

specificity_balbc[cols_balbc].to_csv(Path("Comparison3_Samplewise_Outputs/BALBc_HIL/specificity_analysis_postSYFPEITHI.csv"), index=False)

rank_map_balbc = dict(zip(specificity_balbc["clonotype"], specificity_balbc["rank"]))

plot_all_clonotypes_by_coherence_pdf(
    df=df_balbc_combined, ct_col=CT_COL, markers=markers_balbc, peptides=peptides_balbc,
    out_pdf=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/clonotypes_by_coherence_combined_ranked_postSYFPEITHI.pdf"),
    title_prefix="BALBc_HIL Combined (post-Ly49C + SYFPEITHI) | ranked by specificity order",
    min_size=5, normalize=True, plots_per_page=12,
    rank_map=rank_map_balbc,
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_balbc_combined,
    spec_df=specificity_balbc,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    out_pdf=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/counts_clonotypes_specificity_highlights.pdf"),
    title_prefix="BALBc_HIL Combined",
    normalize=False,
    sample_col=SAMPLE_COL,
    min_prop_threshold=0.05
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_balbc_combined,
    spec_df=specificity_balbc,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    out_pdf=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/prop_clonotypes_specificity_highlights.pdf"),
    title_prefix="BALBc_HIL Combined",
    normalize=True,
    sample_col=SAMPLE_COL,
    min_prop_threshold=0.05
)

plot_correlation_analysis_pdf(
    df=df_balbc_combined,
    spec_df=specificity_balbc,
    ct_col=CT_COL,
    markers=markers_balbc,
    peptides=peptides_balbc,
    out_pdf=Path("Comparison3_Samplewise_Outputs/BALBc_HIL/specificity_correlations.pdf"),
    title_prefix="BALBc_HIL Combined",
)

In [119]:
TCRBETA_COL = "TCR-beta-Tcrb-AMM2021-pAbO"
EEEPVKKI_COL = "RiO-Allo:H-2Kb-EEEPVKKI-pAbO"

output_dir_d7 = Path("Comparison3_Samplewise_Outputs/D7ABC_LL")
output_dir_d7.mkdir(parents=True, exist_ok=True)

df_d7abc_clean = filter_zero_dextramer_cells(df_d7abc, markers_b10br, verbose=True)

after_eeepvkki = {}
for sample, df_s in df_d7abc_clean.groupby(SAMPLE_COL):
    q = chosen_q_eeepvkki_d7abc[sample]
    df_f, Ts = apply_ly49c_filter_one_sample(df_s, EEEPVKKI_COL, q)
    after_eeepvkki[sample] = df_f
    n_ct_before = df_s[CT_COL].nunique()
    n_ct_after = df_f[CT_COL].nunique()
    n_ct_ge5_after = int((df_f[CT_COL].value_counts() >= 5).sum())
    print(f"  {sample}: EEEPVKKI q={q} → {len(df_s):,} → {len(df_f):,} cells | clonotypes: {n_ct_before} → {n_ct_after} ({n_ct_ge5_after} ≥5 cells) (Ts={Ts:.2f})")

df_eeepvkki_combined = pd.concat([after_eeepvkki[s] for s in ["D7A", "D7B", "D7C"]], ignore_index=True)
n_ct_combined_total = df_eeepvkki_combined[CT_COL].nunique()
n_ct_combined_ge5 = int((df_eeepvkki_combined[CT_COL].value_counts() >= 5).sum())
print(f"\nCombined post-EEEPVKKI: {len(df_eeepvkki_combined):,} cells | {n_ct_combined_total} clonotypes total | {n_ct_combined_ge5} clonotypes ≥5 cells")

for sample in ["D7A", "D7B", "D7C"]:
    df_s = after_eeepvkki[sample]
    s_out = output_dir_d7 / sample
    s_out.mkdir(parents=True, exist_ok=True)

    baseline = compute_clonotype_summaries(df_s, CT_COL, markers_b10br, min_size=5, entropy_alpha=1.0)
    base_summ_entropy = {}
    for ct, g in df_s[df_s[CT_COL].isin(set(baseline))].groupby(CT_COL):
        X = g[markers_b10br].to_numpy(dtype=float)
        P = row_normalise(X)
        base_summ_entropy[ct] = {a: float(np.mean(renyi_entropy(P, alpha=float(a), axis=1))) for a in [0,1,2,3,4,5]}
    n0 = len(df_s)

    sweep_rows = []
    sweep_ent_rows = []
    for q in thresholds:
        df_f, Ts = apply_reverse_filter_one_sample(df_s, TCRBETA_COL, float(q))
        filtered = compute_clonotype_summaries(df_f, CT_COL, markers_b10br, min_size=5, entropy_alpha=1.0)
        common = set(baseline) & set(filtered)
        if common:
            cos_d = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "cosine") for c in common]
            l1_d  = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "l1")    for c in common]
            ent_d = [filtered[c]["mean_entropy"] - baseline[c]["mean_entropy"] for c in common]
            coh_d = [filtered[c]["mean_coherence"] - baseline[c]["mean_coherence"] for c in common]
            sweep_rows.append({
                "percentile": float(q), "Ts_tcrbeta": float(Ts),
                "n_cells": len(df_f), "pct_cells_retained": 100.0 * len(df_f) / n0,
                "n_clonotypes_baseline_ge5": len(baseline), "n_clonotypes_filtered_ge5": len(filtered),
                "n_common_clonotypes_ge5": len(common),
                "median_cosine_dist": float(np.median(cos_d)), "median_l1_dist": float(np.median(l1_d)),
                "mean_entropy_change": float(np.mean(ent_d)), "mean_coherence_change": float(np.nanmean(coh_d)),
            })
        else:
            sweep_rows.append({
                "percentile": float(q), "Ts_tcrbeta": float(Ts),
                "n_cells": len(df_f), "pct_cells_retained": 100.0 * len(df_f) / n0,
                "n_clonotypes_baseline_ge5": len(baseline), "n_clonotypes_filtered_ge5": len(filtered),
                "n_common_clonotypes_ge5": 0,
                "median_cosine_dist": np.nan, "median_l1_dist": np.nan,
                "mean_entropy_change": np.nan, "mean_coherence_change": np.nan,
            })
        for a in [0, 1, 2, 3, 4, 5]:
            filt_summ_ent = {}
            for ct, g in df_f[df_f[CT_COL].isin(set(baseline))].groupby(CT_COL):
                if len(g) >= 5:
                    X = g[markers_b10br].to_numpy(dtype=float)
                    P = row_normalise(X)
                    filt_summ_ent[ct] = float(np.mean(renyi_entropy(P, alpha=float(a), axis=1)))
            common_ent = set(base_summ_entropy) & set(filt_summ_ent)
            dH = [filt_summ_ent[ct] - base_summ_entropy[ct][a] for ct in common_ent] if common_ent else [np.nan]
            sweep_ent_rows.append({
                "sample": sample, "percentile": float(q), "Ts_tcrbeta": float(Ts),
                "alpha": float(a), "mean_entropy_change": float(np.mean(dH)),
                "n_common_clonotypes_ge5": len(common_ent), "n_cells_after": len(df_f),
            })

    sweep_tcrbeta = pd.DataFrame(sweep_rows)
    sweep_tcrbeta_entropy = pd.DataFrame(sweep_ent_rows)

    sweep_tcrbeta.to_csv(s_out / "TCRbeta_reverse_sweep_metrics.csv", index=False)
    sweep_tcrbeta_entropy.to_csv(s_out / "TCRbeta_reverse_sweep_entropy.csv", index=False)
    plot_sweep_metrics_2x3(
        sweep_df=sweep_tcrbeta,
        sweep_entropy_df=sweep_tcrbeta_entropy[sweep_tcrbeta_entropy["sample"] == sample],
        out_png=s_out / "TCRbeta_reverse_sweep_metrics_2x3.png",
        title=f"D7ABC LL {sample} (post-EEEPVKKI) | TCR-beta reverse sweep",
        sample_q=None,
    )

chosen_q_tcrbeta_d7abc = {
    "D7A": 0.875,
    "D7B": 0.860,
    "D7C": 0.850,
}

after_tcrbeta = {}
for sample in ["D7A", "D7B", "D7C"]:
    q = chosen_q_tcrbeta_d7abc[sample]
    df_f, Ts = apply_reverse_filter_one_sample(after_eeepvkki[sample], TCRBETA_COL, q)
    after_tcrbeta[sample] = df_f
    n_ct_before = after_eeepvkki[sample][CT_COL].nunique()
    n_ct_after = df_f[CT_COL].nunique()
    n_ct_ge5_after = int((df_f[CT_COL].value_counts() >= 5).sum())
    print(f"  {sample}: TCR-beta top-q={q} → {len(after_eeepvkki[sample]):,} → {len(df_f):,} cells | clonotypes: {n_ct_before} → {n_ct_after} ({n_ct_ge5_after} ≥5 cells) (Ts={Ts:.2f})")

df_d7abc_final = pd.concat([after_tcrbeta[s] for s in ["D7A", "D7B", "D7C"]], ignore_index=True)
n_ct_final_total = df_d7abc_final[CT_COL].nunique()
n_ct_final_ge5 = int((df_d7abc_final[CT_COL].value_counts() >= 5).sum())
print(f"\nCombined post-TCRbeta (df_d7abc_final): {len(df_d7abc_final):,} cells | {n_ct_final_total} clonotypes total | {n_ct_final_ge5} clonotypes ≥5 cells")

[filter_zero_dextramer] Removed 2,739 / 17,991 cells (15.22%) with zero total dextramer counts
  D7A: EEEPVKKI q=0.85 → 5,213 → 4,572 cells | clonotypes: 1679 → 1517 (138 ≥5 cells) (Ts=1.00)
  D7B: EEEPVKKI q=0.9 → 4,787 → 4,332 cells | clonotypes: 1720 → 1598 (156 ≥5 cells) (Ts=2.00)
  D7C: EEEPVKKI q=1.0 → 5,252 → 5,252 cells | clonotypes: 1866 → 1866 (146 ≥5 cells) (Ts=47.00)

Combined post-EEEPVKKI: 14,156 cells | 4902 clonotypes total | 433 clonotypes ≥5 cells
  D7A: TCR-beta top-q=0.875 → 4,572 → 4,022 cells | clonotypes: 1517 → 1413 (124 ≥5 cells) (Ts=4.00)
  D7B: TCR-beta top-q=0.86 → 4,332 → 3,725 cells | clonotypes: 1598 → 1493 (141 ≥5 cells) (Ts=4.34)
  D7C: TCR-beta top-q=0.85 → 5,252 → 4,477 cells | clonotypes: 1866 → 1705 (135 ≥5 cells) (Ts=5.00)

Combined post-TCRbeta (df_d7abc_final): 12,224 cells | 4539 clonotypes total | 392 clonotypes ≥5 cells


In [115]:
specificity_d7abc = full_specificity_analysis(
    df=df_d7abc_final,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    control_peptide="EEEPVKKI",
    min_size=5,
    single_threshold=0.85,
    min_prop_threshold=0.05,
    percentile_threshold=0.95,
    control_fold_threshold=2.0,
)

specificity_d7abc["_class_order"] = specificity_d7abc["classification"].map(class_order)
specificity_d7abc = specificity_d7abc.sort_values(["_class_order", "coherence_full"], ascending=[True, False]).reset_index(drop=True)
specificity_d7abc["rank"] = specificity_d7abc.index + 1
specificity_d7abc = specificity_d7abc.drop(columns=["_class_order"])

cols_d7abc = ["rank", "clonotype"] + [c for c in specificity_d7abc.columns if c not in ["rank", "clonotype", "prop_vector"]]
specificity_d7abc[cols_d7abc].to_csv(output_dir_d7 / "specificity_analysis.csv", index=False)

rank_map_d7abc = dict(zip(specificity_d7abc["clonotype"], specificity_d7abc["rank"]))

plot_all_clonotypes_by_coherence_pdf(
    df=df_d7abc_final, ct_col=CT_COL, markers=markers_b10br, peptides=peptides_b10br,
    out_pdf=output_dir_d7 / "clonotypes_by_coherence_combined_ranked.pdf",
    title_prefix="D7ABC LL (post-Ly49C + EEEPVKKI) | ranked by specificity order",
    min_size=5, normalize=True, plots_per_page=12,
    rank_map=rank_map_d7abc,
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_d7abc_final,
    spec_df=specificity_d7abc,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    out_pdf=output_dir_d7 / "counts_clonotypes_specificity_highlights.pdf",
    title_prefix="D7ABC LL (post-Ly49C + EEEPVKKI)", normalize=False,
    sample_col=SAMPLE_COL,
    min_prop_threshold=0.05,
)

plot_clonotypes_with_specificity_highlights_pdf(
    df=df_d7abc_final,
    spec_df=specificity_d7abc,
    ct_col=CT_COL, markers=markers_b10br,
    peptides=peptides_b10br,
    out_pdf=output_dir_d7 / "prop_clonotypes_specificity_highlights.pdf",
    title_prefix="D7ABC LL (post-Ly49C + EEEPVKKI)",
    normalize=True,
    sample_col=SAMPLE_COL,
    min_prop_threshold=0.05,
)

plot_correlation_analysis_pdf(
    df=df_d7abc_final,
    spec_df=specificity_d7abc,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    out_pdf=output_dir_d7 / "specificity_correlations.pdf",
    title_prefix="D7ABC LL (post-Ly49C + EEEPVKKI)",
)

Plotting 392 clonotypes ordered by decreasing coherence
Saved: Comparison3_Samplewise_Outputs/D7ABC_LL/clonotypes_by_coherence_combined_ranked.pdf
Plotting 392 clonotypes with specificity highlights
Saved: Comparison3_Samplewise_Outputs/D7ABC_LL/counts_clonotypes_specificity_highlights.pdf
Plotting 392 clonotypes with specificity highlights
Saved: Comparison3_Samplewise_Outputs/D7ABC_LL/prop_clonotypes_specificity_highlights.pdf
Plotting correlation analysis for 392 clonotypes
Saved: Comparison3_Samplewise_Outputs/D7ABC_LL/specificity_correlations.pdf


In [116]:
# After EEEPVKKI filter
for s in ["D7A", "D7B", "D7C"]:
    print(f"{s} after EEEPVKKI: {len(after_eeepvkki[s]):,} cells")

# After TCR-beta filter
for s in ["D7A", "D7B", "D7C"]:
    print(f"{s} after TCRbeta: {len(after_tcrbeta[s]):,} cells")

# Final
print(f"df_d7abc_final: {len(df_d7abc_final):,} cells, {df_d7abc_final[CT_COL].nunique()} clonotypes")

D7A after EEEPVKKI: 4,572 cells
D7B after EEEPVKKI: 4,332 cells
D7C after EEEPVKKI: 5,252 cells
D7A after TCRbeta: 4,022 cells
D7B after TCRbeta: 3,725 cells
D7C after TCRbeta: 4,477 cells
df_d7abc_final: 12,224 cells, 4539 clonotypes


In [ ]:
cluster_values = [
    "4 CD200high Deletional tolerance",
    "8 Cycling cells",
    "9 Cycling (trans to CD200high)",
]

cts_in_clusters = set(
    df_d7abc_final[df_d7abc_final["Cluster"].isin(cluster_values)][CT_COL].unique()
)
df_d7abc_clusters = df_d7abc_final[df_d7abc_final[CT_COL].isin(cts_in_clusters)].copy()

print(f"Full dataset: {df_d7abc_final[CT_COL].nunique()} clonotypes, {len(df_d7abc_final):,} cells")
print(f"Cluster subset: {len(cts_in_clusters)} clonotypes, {len(df_d7abc_clusters):,} cells")
print(f"Cluster breakdown:")
for v in cluster_values:
    n_cts   = df_d7abc_clusters[df_d7abc_clusters["Cluster"] == v][CT_COL].nunique()
    n_cells = (df_d7abc_clusters["Cluster"] == v).sum()
    print(f"  {v}: {n_cts} clonotypes, {n_cells:,} cells")

group_dfs_d7abc = {
    "Full LL dataset": df_d7abc_final,
    "Tolerance + Cycling": df_d7abc_clusters[df_d7abc_final["Cluster"].isin(cluster_values)].copy(),
}

plot_static_group_metrics_2x3(
    group_dfs=group_dfs_d7abc,
    ct_col=CT_COL,
    markers=markers_b10br,
    peptides=peptides_b10br,
    out_png=output_dir_d7 / "full_vs_clusters_metrics_2x3.png",
    title="D7ABC LL | Full dataset vs Deletional tolerance + Cycling + Cycling trans CD200high",
    min_size=5,
    renyi_orders=[0, 1, 2, 3, 4, 5],
)

In [ ]:
public_clonotypes_df = pd.read_csv("../Data/Comp 3 - Public clonotypes (HIL and LL).csv")
public_clonotypes_df["HIL_total"] = (
    public_clonotypes_df["BL6-B10BR_HTxA"] +
    public_clonotypes_df["BL6-B10BR_HTxB"] +
    public_clonotypes_df["BL6-B10BR_HTxC"]
)
public_clonotypes_df["LL_total"] = (
    public_clonotypes_df["D7A"] +
    public_clonotypes_df["D7B"] +
    public_clonotypes_df["D7C"]
)

valid_public = public_clonotypes_df[
    (public_clonotypes_df["HIL_total"] >= 1) |
    (public_clonotypes_df["LL_total"]  >= 1)
]
valid_public_clonotypes = set(valid_public["Clonotpe"].tolist())

summaries_hil_raw = compute_clonotype_summaries(df_b10br, CT_COL, markers_b10br, min_size=0)
summaries_ll_raw  = compute_clonotype_summaries(df_d7abc, CT_COL, markers_b10br, min_size=0)

rows = []
for ct in sorted(valid_public_clonotypes):
    in_hil = ct in summaries_hil_raw
    in_ll  = ct in summaries_ll_raw
    if not in_hil and not in_ll:
        continue
    rows.append({
        "clonotype":     ct,
        "n_cells_HIL":   summaries_hil_raw[ct]["n_cells"] if in_hil else 0,
        "n_cells_LL":    summaries_ll_raw[ct]["n_cells"]  if in_ll  else 0,
        "coherence_HIL": summaries_hil_raw[ct]["mean_coherence"] if in_hil else np.nan,
        "coherence_LL":  summaries_ll_raw[ct]["mean_coherence"]  if in_ll  else np.nan,
    })

coherence_comparison = pd.DataFrame(rows)

both_valid  = coherence_comparison.dropna(subset=["coherence_HIL", "coherence_LL"])
hil_only    = coherence_comparison[
    coherence_comparison["coherence_HIL"].notna() &
    coherence_comparison["coherence_LL"].isna()
]
ll_only     = coherence_comparison[
    coherence_comparison["coherence_HIL"].isna() &
    coherence_comparison["coherence_LL"].notna()
]

print(f"Both valid (>=2 cells in both):  {len(both_valid)}")
print(f"HIL only (singleton or absent in LL): {len(hil_only)}")
print(f"LL only  (singleton or absent in HIL): {len(ll_only)}")

output_dir_public = Path("Comparison3_Samplewise_Outputs/HIL_vs_LL_Public")
output_dir_public.mkdir(parents=True, exist_ok=True)
coherence_comparison.to_csv(output_dir_public / "public_clonotypes_coherence_unfiltered.csv", index=False)

all_coh = pd.concat([
    both_valid["coherence_HIL"], both_valid["coherence_LL"],
    hil_only["coherence_HIL"], ll_only["coherence_LL"]
]).dropna()
lo = float(all_coh.min()) - 0.02
hi = float(all_coh.max()) + 0.02

rng = np.random.default_rng(42)
hil_jitter = rng.uniform(lo, hi, len(hil_only))
ll_jitter  = rng.uniform(lo, hi, len(ll_only))

fig, ax = plt.subplots(figsize=(7.5, 7.5))

ax.scatter(
    both_valid["coherence_HIL"], both_valid["coherence_LL"],
    s=55, alpha=0.8, color="steelblue", edgecolors="white", linewidth=0.5,
    label=f"Both ≥2 cells (n={len(both_valid)})", zorder=3
)

ax.scatter(
    hil_only["coherence_HIL"],
    np.full(len(hil_only), lo),
    s=40, alpha=0.6, color="darkorange", edgecolors="white", linewidth=0.5,
    marker="^", label=f"HIL only / LL singleton (n={len(hil_only)})", zorder=3
)

ax.scatter(
    np.full(len(ll_only), lo),
    ll_only["coherence_LL"],
    s=40, alpha=0.6, color="darkgreen", edgecolors="white", linewidth=0.5,
    marker="^", label=f"LL only / HIL singleton (n={len(ll_only)})", zorder=3
)

ax.plot([lo, hi], [lo, hi], "k--", alpha=0.5, lw=1)

ax.axhline(lo, color="gray", lw=0.8, ls=":", alpha=0.6)
ax.axvline(lo, color="gray", lw=0.8, ls=":", alpha=0.6)

if len(both_valid) >= 3:
    x, y = both_valid["coherence_HIL"], both_valid["coherence_LL"]
    r = float(np.corrcoef(x, y)[0, 1])
    ax.text(0.03, 0.97, f"r = {r:.3f}\nN = {len(both_valid)}",
            transform=ax.transAxes, va="top", ha="left", fontsize=10)

ax.set_xlabel("Coherence (HIL, unfiltered)")
ax.set_ylabel("Coherence (LL, unfiltered)")
ax.set_title("Public Clonotypes | HIL vs LL coherence (unfiltered)")
ax.set_xlim(lo - 0.01, hi + 0.01)
ax.set_ylim(lo - 0.01, hi + 0.01)
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(output_dir_public / "public_clonotypes_coherence_HIL_vs_LL_unfiltered.png", dpi=170)
plt.close(fig)
print("Saved")